In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/wikitext103-encoded/train.bin
/kaggle/input/wikitext103-encoded/test.bin
/kaggle/input/wikitext103-encoded/val.bin
/kaggle/input/text8-encoded/train.bin
/kaggle/input/text8-encoded/test.bin
/kaggle/input/text8-encoded/val.bin
/kaggle/input/penn-treebank-encoded/train.bin
/kaggle/input/penn-treebank-encoded/test.bin
/kaggle/input/penn-treebank-encoded/val.bin
/kaggle/input/lambada-encoded/test_dev.bin
/kaggle/input/lambada-encoded/train.bin
/kaggle/input/lambada-encoded/test.bin
/kaggle/input/lambada-encoded/val.bin
/kaggle/input/enwik8-encoded/train.bin
/kaggle/input/enwik8-encoded/test.bin
/kaggle/input/enwik8-encoded/val.bin
/kaggle/input/tinyshakespeare-encoded/train.bin
/kaggle/input/tinyshakespeare-encoded/test.bin
/kaggle/input/tinyshakespeare-encoded/val.bin


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader,IterableDataset
from typing import Tuple, Dict, List, Optional
import numpy as np
from collections import Counter
import requests
import os
import zipfile
import tiktoken
import torch
from torch import Tensor
from tqdm.notebook import tqdm
import time
import math
from torch.nn import functional as F
from torch.amp import GradScaler, autocast
import torch.nn as nn
from torch.optim import lr_scheduler
from functools import partial
from pathlib import Path
from collections import defaultdict
import json
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
n_layer = 6 
n_head = 8

In [3]:
from collections import defaultdict
import torch

class GradientLogger:
    def __init__(self):
        self.data = defaultdict(list)

    def attach(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.register_hook(self._make_hook(name))

    def _make_hook(self, name):
        def hook(grad):
            if grad is not None:
                self.data[name].append(grad.norm().item())
        return hook

    def summary(self, substr=None):
        out = {}
        for k, v in self.data.items():
            if substr is None or substr in k:
                out[k] = sum(v) / max(len(v), 1)
        return out
def gate_entropy(g, eps=1e-6):
    g = g.clamp(eps, 1 - eps)
    return -(g * g.log() + (1 - g) * (1 - g).log()).mean()

def gate_stats(gate, eps=1e-8):
    # gate in (0,1), shape [B, S, D]
    p = gate.clamp(eps, 1 - eps)

    entropy = -(p * p.log() + (1 - p) * (1 - p).log()).mean().item()
    frac_low = (gate < 0.05).float().mean().item()
    frac_high = (gate > 0.95).float().mean().item()
    mean = gate.mean().item()

    return {
        "entropy": entropy,
        "frac_low": frac_low,
        "frac_high": frac_high,
        "mean": mean,
    }
def collect_gate_entropy(model):
    entropies = []
    for m in model.modules():
        if isinstance(m, RemixedLinear) and hasattr(m, "last_gate_entropy"):
            entropies.append(m.last_gate_entropy)
    if len(entropies) == 0:
        return None
    return torch.stack(entropies).mean()

def collect_gate_stats(model):
    stats = []
    for m in model.modules():
        if isinstance(m, RemixedLinear) and hasattr(m, "last_gate_stats"):
            stats.append(m.last_gate_stats)
    return stats
class TemperatureAnnealer:
    def __init__(
        self,
        start=1.5,
        end=0.2,
        total_steps=100_000,
        mode="cosine"  # or "linear", "exp"
    ):
        self.start = start
        self.end = end
        self.total_steps = total_steps
        self.mode = mode

    def __call__(self, step):
        t = min(step / self.total_steps, 1.0)

        if self.mode == "linear":
            tau = self.start + t * (self.end - self.start)

        elif self.mode == "cosine":
            tau = self.end + 0.5 * (self.start - self.end) * (1 + math.cos(math.pi * t))

        elif self.mode == "exp":
            tau = self.start * (self.end / self.start) ** t

        else:
            raise ValueError(self.mode)

        return max(tau, 1e-3)  # HARD FLOOR


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. The Core Router (Your provided class)
# ==========================================
class ImprovedContextAwareRouter(nn.Module):
    """Enhanced router with vocabulary-aware initialization and better scaling."""
    def __init__(
        self,
        vocab_size,
        num_experts, 
        router_dim,
        full_embed_dim,
        context_window=8,
        causal=True,
        num_heads=4,
        num_queries=8,
        n_layers=1,
        use_vocab_prior=True,
    ):
        super().__init__()
        # In this specific config, num_experts will equal target_dim (64)
        self.num_experts = num_experts 
        self.router_dim = router_dim
        self.context_window = context_window
        self.causal = causal
        self.num_heads = num_heads
        self.head_dim = router_dim // num_heads
        self.num_queries = num_queries
        self.n_layers = n_layers
        self.use_vocab_prior = use_vocab_prior
        
        # Project full embeddings to router dimension
        self.embed_proj = nn.Linear(full_embed_dim, router_dim)
        
        # Multi-head attention
        self.qkv_proj = nn.Linear(router_dim, 3 * router_dim, bias=False)
        self.out_proj = nn.Linear(router_dim, router_dim)
        
        # Expert projection with layer norm for stability
        self.ln = nn.LayerNorm(router_dim)
        self.routing_queries = nn.Parameter(torch.randn(num_queries, router_dim))
        self.temperature_predictor = nn.Linear(router_dim, 1)
        
        self.expert_proj = nn.Linear(router_dim, num_experts)
        self.cross_expert_proj = nn.Linear(router_dim, num_experts)
        self.alpha_gate = nn.Linear(router_dim, 1)
        
        # Vocabulary prior for better initialization
        if use_vocab_prior:
            self.vocab_routing_bias = nn.Embedding(vocab_size, num_experts)
            nn.init.normal_(self.vocab_routing_bias.weight, mean=0, std=0.02)
        
        # Better initialization for large vocab
        nn.init.normal_(self.expert_proj.weight, mean=0, std=0.02)
        nn.init.normal_(self.cross_expert_proj.weight, mean=0, std=0.02)
        
    def forward(self, full_embeds, input_ids=None):
        batch_size, seq_len, _ = full_embeds.shape
        
        # Project to router dimension
        x = self.embed_proj(full_embeds)
        
        # Multi-head context attention
        for _ in range(self.n_layers):
            x = self._multi_head_attention(x)
        
        # Layer norm before expert projection
        x = self.ln(x)
        
        # Get expert logits
        self_attn_logits = self.expert_proj(x)
        cross_attn_logits = self.cross_expert_proj(self._cross_attention(self.routing_queries, x))
        
        alpha = torch.sigmoid(self.alpha_gate(x))
        expert_logits = alpha * self_attn_logits + (1 - alpha) * cross_attn_logits
        
        # Add vocabulary prior if available
        if self.use_vocab_prior and input_ids is not None:
            vocab_bias = self.vocab_routing_bias(input_ids)
            expert_logits = expert_logits + vocab_bias
        
        adaptive_temp = torch.sigmoid(self.temperature_predictor(x)) * 2.0 + 0.1
        
        # We return expert_logits as the embedding itself
        return expert_logits, adaptive_temp, self.expert_proj.weight

    def _cross_attention(self, queries, context):
        batch_size, seq_len, dim = context.shape
        num_queries = queries.shape[0]
        
        queries = queries.unsqueeze(0).expand(batch_size, num_queries, dim)
        attn_scores = torch.matmul(queries, context.transpose(1, 2)) / (dim ** 0.5)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attended = torch.matmul(attn_weights, context)
        fused = attended.mean(dim=1).unsqueeze(1).expand(-1, seq_len, -1)
        return fused
    
    def _multi_head_attention(self, x):
        batch_size, seq_len, _ = x.shape
        
        qkv = self.qkv_proj(x).reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        mask = self._create_mask(seq_len, x.device).unsqueeze(0).unsqueeze(0)
        attn_scores = attn_scores.masked_fill(mask, float('-inf'))
        
        attn_weights = F.softmax(attn_scores, dim=-1)
        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).reshape(batch_size, seq_len, self.router_dim)
        output = self.out_proj(context) + x
        return output
    
    def _create_mask(self, seq_len, device):
        if self.context_window == -1:
            if self.causal:
                positions = torch.arange(seq_len, device=device)
                pos_diff = positions.unsqueeze(0) - positions.unsqueeze(1)
                return pos_diff > 0
            else:
                return torch.zeros(seq_len, seq_len, dtype=torch.bool, device=device)
        
        positions = torch.arange(seq_len, device=device)
        pos_diff = positions.unsqueeze(0) - positions.unsqueeze(1)
        
        if self.causal:
            return (pos_diff > 0) | (pos_diff < -self.context_window)
        else:
            window_half = self.context_window // 2
            return torch.abs(pos_diff) > window_half


# ==========================================
# 2. Pure Router Remix (64 -> 64)
# ==========================================
class DirectContextualEmbedding(nn.Module):
    """
    Directly uses the Router to transform a static 64-dim seed into 
    a dynamic 64-dim embedding. No experts, no MoE weighted sums.
    """
    def __init__(
        self, 
        vocab_size, 
        dim=64,             # Staying in 64 dimensions
        context_window=32,
        dropout=0.0
    ):
        super().__init__()
        
        # 1. Static Seed (The "Base" meaning)
        # Size: [Vocab, 64]
        self.seed_embeddings = nn.Embedding(vocab_size, dim)
        
        # 2. The Router (The "Remixer")
        # - Inputs 64 dims
        # - "Thinks" in 64 dims
        # - Outputs 64 "logits" -> These ARE the new embedding
        self.router = ImprovedContextAwareRouter(
            vocab_size=vocab_size,
            num_experts=dim,    # CRITICAL: We want exactly 64 output values
            router_dim=dim,     # Internal thinking size
            full_embed_dim=dim, # Input size
            context_window=context_window,
            causal=True,
            n_layers=2,         # Depth of reasoning
            use_vocab_prior=False
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # 3. Norm ensures the router output behaves like a stable embedding
        self.out_norm = nn.LayerNorm(dim)

    def forward(self, input_ids):
        # 1. Get Static Seed
        # [B, S, 64]
        seeds = self.seed_embeddings(input_ids)
        
        # 2. Remix via Router
        # The router returns (logits, temp, weights). 
        # Here, 'logits' is our 64-dim vector.
        # It has looked at the surrounding tokens to decide these values.
        remixed_embeds, _, _ = self.router(seeds, input_ids)
        
        # 3. Residual Connection (Optional but recommended)
        # Keeps the base meaning while adding context
        output = self.dropout(remixed_embeds)# +seeds
        
        return self.out_norm(output)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# ==========================================
# 3. Parameter Efficiency Verification
# ==========================================
if __name__ == "__main__":
    vocab_size = 50257
    dim = 64  # The target constraint
    
    print("="*60)
    print("COMPARISON: Standard 512 vs Remix 64")
    print("="*60)
    
    # --- 1. Baseline (Standard Dense 512) ---
    baseline_emb = nn.Embedding(vocab_size, 512)
    base_params = count_parameters(baseline_emb)
    print(f"Baseline (512-dim Dense):")
    print(f"  Params: {base_params:,}")
    print(f"  Memory: {base_params * 4 / 1024 / 1024:.2f} MB")
    
    # --- 2. Pure Remix (64 -> 64) ---
    remix_model = DirectContextualEmbedding(vocab_size, dim=64)
    remix_params = count_parameters(remix_model)
    
    # Calculate Router Overhead
    seed_params = vocab_size * 64
    router_overhead = remix_params - seed_params
    
    print(f"\nPure Remix (64-dim Contextual):")
    print(f"  Total Params: {remix_params:,}")
    print(f"    - Seed Table: {seed_params:,}")
    print(f"    - Router Overhead: {router_overhead:,} (Only ~0.3M params!)")
    print(f"  Memory: {remix_params * 4 / 1024 / 1024:.2f} MB")
    
    # --- 3. The Result ---
    ratio = base_params / remix_params
    print(f"\nFINAL STATS:")
    print(f"  Compression Ratio: {ratio:.2f}x smaller than Baseline")
    print(f"  Output Dimension: {dim} (As requested)")
    
    # --- 4. Forward Check ---
    dummy = torch.randint(0, vocab_size, (2, 32))
    out = remix_model(dummy)
    print(f"\nOutput Shape: {out.shape} -> Correct")

COMPARISON: Standard 512 vs Remix 64
Baseline (512-dim Dense):
  Params: 25,731,584
  Memory: 98.16 MB

Pure Remix (64-dim Contextual):
  Total Params: 3,246,274
    - Seed Table: 3,216,448
    - Router Overhead: 29,826 (Only ~0.3M params!)
  Memory: 12.38 MB

FINAL STATS:
  Compression Ratio: 7.93x smaller than Baseline
  Output Dimension: 64 (As requested)

Output Shape: torch.Size([2, 32, 64]) -> Correct


In [5]:

class PermutationMoE(nn.Module):
    """
    Learns context-aware permutations (with optional replacement) of a base embedding.
    Instead of selecting from 512 dims, we start with 64 dims and learn optimal
    permutations/selections with repetition.
    """
    def __init__(
        self,
        vocab_size,
        block_size=64,
        base_embed_dim=64,  # Start with smaller embedding
        num_experts=8,
        router_dim=64,
        selection_mode='soft',  # 'soft' (differentiable) or 'hard' (Gumbel)
        use_dim_mlp_refinement=True,
        mlp_refinement_position='after',  # 'before' or 'after' selection
        allow_replacement=True,  # If True, dims can be repeated
        temperature=1.0,
        dropout=0.0,
        load_balance_weight=0.01,
        z_loss_weight=0.001,
        diversity_weight=0.01,
        # Router params
        context_window=-1,
        causal=True,
        num_queries=8,
        n_layers=2,
        use_vocab_prior=False,
        num_heads=8,
    ):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.base_embed_dim = base_embed_dim
        self.num_experts = num_experts
        self.selection_mode = selection_mode
        self.use_dim_mlp_refinement = use_dim_mlp_refinement
        self.mlp_refinement_position = mlp_refinement_position
        self.allow_replacement = allow_replacement
        self.load_balance_weight = load_balance_weight
        self.z_loss_weight = z_loss_weight
        self.diversity_weight = diversity_weight
        
        # Base embeddings (much smaller than 512)
        self.embeddings = nn.Embedding(vocab_size, base_embed_dim)
        self.position_embeddings = nn.Embedding(block_size, base_embed_dim)
        
        # Each expert learns a context-dependent dimension selector
        # For each output dimension, select/weight from input dimensions
        self.dim_selectors = nn.ModuleList([
            nn.Sequential(
                nn.Linear(base_embed_dim, router_dim),
                nn.LayerNorm(router_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                # Output: for each output dim, logits over input dims
                nn.Linear(router_dim, base_embed_dim * base_embed_dim),
            ) for _ in range(num_experts)
        ])
        
        # Optional MLP refinement
        if use_dim_mlp_refinement:
            if mlp_refinement_position == 'before':
                # Single shared MLP before expert selection
                self.refinement_mlp = nn.Sequential(
                    nn.LayerNorm(base_embed_dim),
                    nn.Linear(base_embed_dim, base_embed_dim * 2),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(base_embed_dim * 2, base_embed_dim)
                )
            else:  # 'after'
                # Per-expert MLPs after selection
                self.expert_mlps = nn.ModuleList([
                    nn.Sequential(
                        nn.LayerNorm(base_embed_dim),
                        nn.Linear(base_embed_dim, base_embed_dim * 2),
                        nn.GELU(),
                        nn.Dropout(dropout),
                        nn.Linear(base_embed_dim * 2, base_embed_dim)
                    ) for _ in range(num_experts)
                ])
        
        # Expert router
        self.expert_router = ImprovedContextAwareRouter(
            vocab_size=vocab_size,
            num_experts=num_experts,
            router_dim=router_dim,
            full_embed_dim=base_embed_dim,
            context_window=context_window,
            causal=causal,
            num_heads=num_heads,
            num_queries=num_queries,
            n_layers=n_layers,
            use_vocab_prior=use_vocab_prior,
        )
        
        self.ln = nn.LayerNorm(base_embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('temperature', torch.tensor(temperature))
        
        # # Initialize selectors to favor identity (no permutation initially)
        # for module in self.dim_selectors:
        #     last_layer = module[-1]
        #     # Bias toward identity permutation
        #     identity_flat = torch.eye(base_embed_dim).flatten()
        #     last_layer.bias.data.copy_(identity_flat * 2.0)
        #     nn.init.normal_(last_layer.weight, mean=0, std=0.02)
    
    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        device = input_ids.device
        positions = torch.arange(seq_len, device=device)
        
        # Get base embeddings [B, L, D]
        embeds = self.embeddings(input_ids) + self.position_embeddings(positions)
        
        # Optional refinement before selection
        if self.use_dim_mlp_refinement and self.mlp_refinement_position == 'before':
            embeds = embeds + self.refinement_mlp(embeds)
        
        # Each expert generates context-aware permuted embeddings
        expert_outputs = []
        selection_patterns = []
        
        for expert_idx in range(self.num_experts):
            # Generate selection matrix based on context
            # Use per-position context to decide dimension selection
            selection_logits = self.dim_selectors[expert_idx](embeds)
            # [B, L, D*D] -> [B, L, D_out, D_in]
            selection_logits = selection_logits.view(
                batch_size, seq_len, self.base_embed_dim, self.base_embed_dim
            )
            
            # Apply selection based on mode
            if self.selection_mode == 'soft':# or self.allow_replacement:
                # print("soft")
                # Soft attention: each output dim is weighted sum of input dims
                # This naturally allows replacement
                selection_weights = F.softmax(
                    selection_logits / self.temperature, dim=-1
                )  # [B, L, D_out, D_in]
                
                # For each position, apply the selection matrix
                # embeds: [B, L, D_in]
                # selection_weights: [B, L, D_out, D_in]
                selected = torch.einsum('bloi,bli->blo', selection_weights, embeds)
                # [B, L, D_out]
                
            elif self.selection_mode == 'hard' and not self.allow_replacement:
                # Hard selection without replacement (true permutation)
                # Use Gumbel-Softmax with straight-through estimator
                selection_weights = F.gumbel_softmax(
                    selection_logits, tau=self.temperature, hard=True, dim=-1
                )  # [B, L, D_out, D_in]
                
                selected = torch.einsum('bloi,bli->blo', selection_weights, embeds)
            
            else:  # Hard with replacement
                # Sample hard indices but allow duplicates
                selection_weights = F.gumbel_softmax(
                    selection_logits, tau=self.temperature, hard=False, dim=-1
                )
                selected = torch.einsum('bloi,bli->blo', selection_weights, embeds)
            
            # Optional refinement after selection
            if self.use_dim_mlp_refinement and self.mlp_refinement_position == 'after':
                selected = selected + self.expert_mlps[expert_idx](selected)
            
            expert_outputs.append(selected)
            selection_patterns.append(selection_weights)
        
        # Stack expert outputs [B, L, num_experts, D]
        expert_outputs = torch.stack(expert_outputs, dim=2)
        selection_patterns = torch.stack(selection_patterns, dim=2)  # [B, L, E, D, D]
        
        # Route among experts based on context
        expert_logits, adaptive_temp, _ = self.expert_router(embeds, input_ids)
        expert_weights = F.softmax(expert_logits / (self.temperature * adaptive_temp), dim=-1)
        expert_weights = self.dropout(expert_weights)
        
        # Combine expert outputs
        output = (expert_weights.unsqueeze(-1) * expert_outputs).sum(dim=2)
        output = self.ln(output)
        
        # Compute auxiliary losses
        aux_losses = self._compute_aux_losses(
            expert_logits, expert_weights, selection_patterns
        )
        
        routing_info = {
            'expert_logits': expert_logits,
            'expert_weights': expert_weights,
            'selection_patterns': selection_patterns,
            'expert_usage': expert_weights.mean(dim=[0, 1]),
            'aux_losses': aux_losses,
            'total_aux_loss': sum(aux_losses.values())
        }
        
        return output, routing_info
    
    def _compute_aux_losses(self, expert_logits, expert_weights, selection_patterns):
        losses = {}
        
        # Expert load balancing
        expert_usage = expert_weights.sum(dim=[0, 1])
        mean_usage = expert_usage.mean()
        losses['load_balance'] = self.load_balance_weight * \
            (expert_usage - mean_usage).pow(2).mean()
        
        # Z-loss for routing stability
        losses['z_loss'] = self.z_loss_weight * \
            torch.logsumexp(expert_logits, dim=-1).pow(2).mean()
        
        # Diversity loss: encourage experts to learn different selection patterns
        # Average selection pattern per expert across batch and sequence
        avg_patterns = selection_patterns.mean(dim=[0, 1])  # [E, D_out, D_in]
        # Flatten to [E, D*D]
        avg_patterns_flat = avg_patterns.reshape(self.num_experts, -1)
        # Normalize and compute correlation
        normed = F.normalize(avg_patterns_flat, dim=-1)
        correlation = torch.matmul(normed, normed.T)
        # Penalize high correlation (exclude diagonal)
        mask = ~torch.eye(self.num_experts, device=correlation.device, dtype=torch.bool)
        diversity_loss = correlation[mask].pow(2).mean()
        losses['diversity'] = self.diversity_weight * diversity_loss
        
        return losses


def count_parameters(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def print_model_info(name, model, output, info):
    """Print model statistics."""
    total, trainable = count_parameters(model)
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"Output shape: {output.shape}")
    print(f"Total params: {total:,}")
    print(f"Trainable params: {trainable:,}")
    print(f"Expert usage: {info['expert_usage'].tolist()}")
    print(f"Aux losses: {info['aux_losses']}")
    print(f"Total aux loss: {info['total_aux_loss']:.4f}")


# Testing
if __name__ == "__main__":
    vocab_size = 50257
    batch_size = 4
    seq_len = 32
    
    print("\n" + "="*60)
    print("TESTING PERMUTATION-BASED MOE VARIANTS")
    print("="*60)
    
    input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
    
    # Test 1: Soft selection with replacement (default)
    print("\n" + "="*60)
    print("Test 1: Soft Selection WITH Replacement")
    print("- Allows dimension repetition")
    print("- MLP refinement AFTER selection")
    print("="*60)
    model1 = PermutationMoE(
        vocab_size=vocab_size,
        base_embed_dim=64,
        num_experts=8,
        selection_mode='soft',
        allow_replacement=True,
        use_dim_mlp_refinement=True,
        mlp_refinement_position='after',
    )
    output1, info1 = model1(input_ids)
    print_model_info("Soft Selection (with replacement, MLP after)", model1, output1, info1)
    
    # Test 2: Soft selection WITHOUT replacement
    print("\n" + "="*60)
    print("Test 2: Soft Selection WITHOUT Replacement")
    print("- Encourages true permutations")
    print("- MLP refinement BEFORE selection")
    print("="*60)
    model2 = PermutationMoE(
        vocab_size=vocab_size,
        base_embed_dim=64,
        num_experts=8,
        selection_mode='soft',
        allow_replacement=False,
        use_dim_mlp_refinement=True,
        mlp_refinement_position='before',
    )
    output2, info2 = model2(input_ids)
    print_model_info("Soft Selection (no replacement, MLP before)", model2, output2, info2)
    
    # Test 3: Hard selection with Gumbel
    print("\n" + "="*60)
    print("Test 3: Hard Selection (Gumbel) WITH Replacement")
    print("- Discrete sampling via Gumbel-Softmax")
    print("- No MLP refinement")
    print("="*60)
    model3 = PermutationMoE(
        vocab_size=vocab_size,
        base_embed_dim=64,
        num_experts=8,
        selection_mode='hard',
        allow_replacement=True,
        use_dim_mlp_refinement=False,
    )
    output3, info3 = model3(input_ids)
    print_model_info("Hard Selection (Gumbel, with replacement, no MLP)", model3, output3, info3)
    
    # Test 4: No MLP, soft selection
    print("\n" + "="*60)
    print("Test 4: Minimal Configuration")
    print("- Soft selection, replacement allowed")
    print("- No MLP refinement (fastest)")
    print("="*60)
    model4 = PermutationMoE(
        vocab_size=vocab_size,
        base_embed_dim=64,
        num_experts=4,  # Fewer experts
        selection_mode='soft',
        allow_replacement=True,
        use_dim_mlp_refinement=False,
    )
    output4, info4 = model4(input_ids)
    print_model_info("Minimal (4 experts, no MLP)", model4, output4, info4)
    
    # Parameter comparison
    print("\n" + "="*60)
    print("PARAMETER COMPARISON")
    print("="*60)
    models = [
        ("Soft + Replacement + MLP(after)", model1),
        ("Soft + No Replace + MLP(before)", model2),
        ("Hard(Gumbel) + Replace + No MLP", model3),
        ("Minimal (4 experts)", model4),
    ]
    
    print(f"{'Model':<35} {'Total Params':>15} {'Non-Embed':>15}")
    print("-"*70)
    for name, model in models:
        total, _ = count_parameters(model)
        embed_params = sum(p.numel() for pname, p in model.named_parameters() 
                          if 'embedding' in pname.lower())
        print(f"{name:<35} {total:>15,} {total-embed_params:>15,}")
    
    # Analyze selection patterns - PROPER REPLACEMENT ANALYSIS
    print("\n" + "="*60)
    print("SELECTION PATTERN ANALYSIS (Model 1)")
    print("="*60)
    patterns = info1['selection_patterns'][0, 0]  # First batch, first position
    print(f"Shape: {patterns.shape} [num_experts, D_out, D_in]")
    
    for expert_idx in range(min(3, model1.num_experts)):
        pattern = patterns[expert_idx]  # [D_out, D_in]
        
        # PROPER ANALYSIS: Sum of weights each input dim receives
        input_dim_usage = pattern.sum(dim=0)  # Sum across output dims [D_in]
        
        # Statistics
        max_usage = input_dim_usage.max().item()
        min_usage = input_dim_usage.min().item()
        mean_usage = input_dim_usage.mean().item()
        std_usage = input_dim_usage.std().item()
        
        # Check for replacement: are some dims used >1.0x?
        heavily_used = (input_dim_usage > 1.2).sum().item()
        barely_used = (input_dim_usage < 0.8).sum().item()
        
        # Entropy of selection (lower = more concentrated)
        pattern_flat = pattern.flatten()
        entropy = -(pattern_flat * torch.log(pattern_flat + 1e-10)).sum().item()
        
        # Top input dims by total usage
        top_vals, top_indices = input_dim_usage.topk(5)
        bottom_vals, bottom_indices = input_dim_usage.topk(5, largest=False)
        
        print(f"\nExpert {expert_idx}:")
        print(f"  Input dim usage statistics:")
        print(f"    Mean: {mean_usage:.3f} (expect ~1.0 for uniform)")
        print(f"    Std:  {std_usage:.3f} (higher = more specialization)")
        print(f"    Max:  {max_usage:.3f} (>1.0 indicates replacement)")
        print(f"    Min:  {min_usage:.3f} (<1.0 indicates some dims ignored)")
        print(f"  Replacement evidence:")
        print(f"    Dims used >1.2x: {heavily_used}/64 (replicated)")
        print(f"    Dims used <0.8x: {barely_used}/64 (under-utilized)")
        print(f"  Selection entropy: {entropy:.2f} (lower = more concentrated)")
        print(f"  Top 5 input dims: {top_indices.tolist()}")
        print(f"    Usage: {top_vals.tolist()}")
        print(f"  Bottom 5 input dims: {bottom_indices.tolist()}")
        print(f"    Usage: {bottom_vals.tolist()}")
    
    print("\n" + "="*60)
    print("✅ All tests passed!")
    print(f"Peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB" 
          if torch.cuda.is_available() else "CPU mode")
    print("="*60)


TESTING PERMUTATION-BASED MOE VARIANTS

Test 1: Soft Selection WITH Replacement
- Allows dimension repetition
- MLP refinement AFTER selection

Soft Selection (with replacement, MLP after)
Output shape: torch.Size([4, 32, 64])
Total params: 5,540,946
Trainable params: 5,540,946
Expert usage: [0.13273143768310547, 0.1276526302099228, 0.11820174753665924, 0.1328621804714203, 0.11222454160451889, 0.1277965009212494, 0.12425566464662552, 0.12427528202533722]
Aux losses: {'load_balance': tensor(0.0071, grad_fn=<MulBackward0>), 'z_loss': tensor(0.0045, grad_fn=<MulBackward0>), 'diversity': tensor(0.0093, grad_fn=<MulBackward0>)}
Total aux loss: 0.0209

Test 2: Soft Selection WITHOUT Replacement
- Encourages true permutations
- MLP refinement BEFORE selection

Soft Selection (no replacement, MLP before)
Output shape: torch.Size([4, 32, 64])
Total params: 5,424,018
Trainable params: 5,424,018
Expert usage: [0.11979085206985474, 0.1197608932852745, 0.12937600910663605, 0.1317342072725296, 0.12

In [6]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# # --- 1. The Core Component: Remixed Linear Expert ---
# class RemixedExpert(nn.Module):
#     """
#     The 'Context-Aware Basis' Layer.
#     Replaces the heavy "generate a whole matrix" approach.
#     """
#     def __init__(self, in_features, out_features, context_dim, basis_size=64):
#         super().__init__()
#         self.basis_size = basis_size
        
#         # A. The Basis Bank (Input -> Basis Features)
#         # "Here are the 64 fundamental ingredients we can use"
#         self.basis = nn.Linear(in_features, basis_size, bias=False)
        
#         # CRITICAL FIX: Normalize basis before gating to prevent "loud" features leaking
#         self.ln_basis = nn.LayerNorm(basis_size)
        
#         # B. The Template Mixing (Basis -> Output)
#         # "Here is the standard recipe for combining ingredients"
#         self.template_mixing = nn.Parameter(torch.randn(out_features, basis_size))
        
#         # C. The Context Controller (Input -> Gates)
#         # "Decide which ingredients and which recipes to use right now"
#         # Outputs: [Basis_Gate (64) + Output_Gate (512)]
#         self.context_modulator = nn.Sequential(
#             nn.Linear(context_dim, basis_size), # Compress context
#             nn.GELU(),
#             nn.Linear(basis_size, basis_size + out_features),
#             nn.Sigmoid() # Soft Gating (0.0 to 1.0)
#         )
        
#         self.bias = nn.Parameter(torch.zeros(out_features))
#         self.reset_parameters()
#         self.final_ln = nn.Linear(out_features, out_features)

#     def reset_parameters(self):
#         nn.init.orthogonal_(self.basis.weight)
#         nn.init.kaiming_normal_(self.template_mixing)
#         # Initialize gates to be mostly OPEN (bias=2.0 -> sigmoid=0.88)
#         # This prevents "dead expert" start
#         nn.init.constant_(self.context_modulator[-2].bias, 2.0) 

#     def forward(self, x, context):
#         # x: [Batch, Seq, In]
        
#         # 1. Project to Basis & Normalize
#         h_basis = self.basis(x)
#         h_basis = self.ln_basis(h_basis) # The "Loudness" Fix
        
#         # 2. Generate Gates from Context
#         gates = self.context_modulator(context)
#         gate_basis = gates[..., :self.basis_size]
#         gate_out = gates[..., self.basis_size:]
        
#         # 3. Apply Basis Gating (Select features)
#         h_gated = h_basis * gate_basis
        
#         # 4. Apply Template Mixing (The Remix)
#         # F.linear uses the weight matrix (Output, Input), so we use standard broadcast
#         pre_output = F.linear(h_gated, self.template_mixing)
        
#         # 5. Apply Output Gating (Select active neurons)
#         output = (pre_output * gate_out) + self.bias
#         output = self.final_ln(output)# + output
        
        
#         return output , gates

# # --- 2. The Main Module: Permutation MOE ---
# class PermutationMoE(nn.Module):
#     """
#     Revised MoE using Remixed Basis Experts.
#     """
#     def __init__(
#         self,
#         vocab_size,
#         block_size=64,
#         base_embed_dim=64,
#         num_experts=8,
#         router_dim=64,
#         context_window=-1,
#         causal=True,
#         num_queries=8,
#         n_layers=2,
#         use_vocab_prior=False,
#         num_heads=8,
#         # Legacy params we might ignore now but keep for compatibility
#         selection_mode='soft', 
#         allow_replacement=True,
#         dropout=0.0,
#         load_balance_weight=0.01,
#         z_loss_weight=0.001,
#         diversity_weight=0.01,
#         use_dim_mlp_refinement=True,
#         mlp_refinement_position='after',  # 'before' or 'after' selection
        
#     ):
#         super().__init__()
        
#         self.base_embed_dim = base_embed_dim
#         self.num_experts = num_experts
#         self.load_balance_weight = load_balance_weight
#         self.z_loss_weight = z_loss_weight
#         self.diversity_weight = diversity_weight
        
#         # 1. Embeddings
#         self.embeddings = nn.Embedding(vocab_size, base_embed_dim)
#         self.position_embeddings = nn.Embedding(block_size, base_embed_dim)
        
#         # 2. The Experts (Now RemixedLinear instances)
#         # Note: We treat 'x' and 'context' as the same input (self-driven)
#         self.experts = nn.ModuleList([
#             RemixedExpert(
#                 in_features=base_embed_dim,
#                 out_features=base_embed_dim,
#                 context_dim=base_embed_dim,
#                 basis_size=base_embed_dim # Keeping basis size same as embed dim for 1:1 mapping potential
#             ) for _ in range(num_experts)
#         ])
        
#         # 3. Router (Assuming ImprovedContextAwareRouter exists in your scope)
#         # For this snippet to run standalone, I'll mock a simple router if that class is missing
#         try:
#             self.expert_router = ImprovedContextAwareRouter(
#                 vocab_size=vocab_size, num_experts=num_experts, router_dim=router_dim,
#                 full_embed_dim=base_embed_dim, context_window=context_window, causal=causal,
#                 num_heads=num_heads, num_queries=num_queries, n_layers=n_layers,
#                 use_vocab_prior=use_vocab_prior,
#             )
#         except NameError:
#             print("Warning: ImprovedContextAwareRouter not found. Using simple Linear Router.")
#             self.expert_router = nn.Linear(base_embed_dim, num_experts)
#             self.simple_router = True
#         else:
#             self.simple_router = False

#         self.ln = nn.LayerNorm(base_embed_dim)
#         self.dropout = nn.Dropout(dropout)
#         self.register_buffer('temperature', torch.tensor(1.0))

#     def forward(self, input_ids):
#         batch_size, seq_len = input_ids.shape
#         device = input_ids.device
#         positions = torch.arange(seq_len, device=device)
        
#         # 1. Base Embeddings
#         embeds = self.embeddings(input_ids) + self.position_embeddings(positions)
        
#         # 2. Run Experts
#         # Each expert acts as a "Remixer"
#         expert_outputs = []
#         all_gates = [] # To track patterns for diversity loss
        
#         for expert in self.experts:
#             # Pass (x, context) -> in this case, both are 'embeds'
#             out, gates = expert(embeds, embeds) 
#             expert_outputs.append(out)
#             all_gates.append(gates)
            
#         # Stack: [B, L, Num_Experts, D]
#         expert_outputs = torch.stack(expert_outputs, dim=2)
#         # Stack Gates: [B, L, Num_Experts, Gate_Dim]
#         all_gates = torch.stack(all_gates, dim=2)

#         # 3. Routing
#         if not self.simple_router:
#             expert_logits, adaptive_temp, _ = self.expert_router(embeds, input_ids)
#             expert_weights = F.softmax(expert_logits / (self.temperature * adaptive_temp), dim=-1)
#         else:
#             expert_logits = self.expert_router(embeds)
#             expert_weights = F.softmax(expert_logits, dim=-1)
            
#         expert_weights = self.dropout(expert_weights)
        
#         # 4. Combine (Weighted Sum)
#         # [B, L, E, 1] * [B, L, E, D] -> Sum over E
#         output = (expert_weights.unsqueeze(-1) * expert_outputs).sum(dim=2)
#         output = self.ln(output)
        
#         # 5. Aux Losses
#         aux_losses = self._compute_aux_losses(expert_logits, expert_weights, all_gates)
        
#         routing_info = {
#             'expert_usage': expert_weights.mean(dim=[0, 1]),
#             'aux_losses': aux_losses,
#             'total_aux_loss': sum(aux_losses.values())
#         }
        
#         return output, routing_info

#     def _compute_aux_losses(self, expert_logits, expert_weights, all_gates):
#         losses = {}
        
#         # A. Load Balancing (Standard)
#         expert_usage = expert_weights.sum(dim=[0, 1])
#         mean_usage = expert_usage.mean()
#         losses['load_balance'] = self.load_balance_weight * (expert_usage - mean_usage).pow(2).mean()
        
#         # B. Z-Loss (Router Stability)
#         losses['z_loss'] = self.z_loss_weight * torch.logsumexp(expert_logits, dim=-1).pow(2).mean()
        
#         # C. Diversity Loss (New: Based on Gates)
#         # We want experts to use DIFFERENT basis features.
#         # all_gates: [B, L, E, Gate_Dim]
#         # Average gate activation per expert: [E, Gate_Dim]
#         avg_gates = all_gates.mean(dim=[0, 1])
        
#         # Normalize to compare shapes of gate vectors
#         normed = F.normalize(avg_gates, dim=-1)
#         correlation = torch.matmul(normed, normed.T) # [E, E]
        
#         # Penalize off-diagonal correlations (experts looking identical)
#         mask = ~torch.eye(self.num_experts, device=correlation.device, dtype=torch.bool)
#         losses['diversity'] = self.diversity_weight * correlation[mask].pow(2).mean()
        
#         return losses

# # --- Testing Block (Copy-Paste this to verify) ---
# if __name__ == "__main__":
#     vocab_size = 50257
#     model = PermutationMoE(vocab_size=vocab_size, base_embed_dim=64, num_experts=4)
    
#     # Dummy Input
#     x = torch.randint(0, vocab_size, (2, 32))
    
#     # Forward
#     out, info = model(x)
    
#     print("Output Shape:", out.shape)
#     print("Expert Usage:", info['expert_usage'])
#     print("Total Aux Loss:", info['total_aux_loss'])
    
#     # Verify the Gates are moving (The Pulse Check)
#     # We access the first expert to see if gates are 0.5 or varied
#     first_expert_gates = model.experts[0].context_modulator(model.embeddings(x))
#     print(f"Gate Std (Should be > 0.2): {first_expert_gates.std().item():.4f}")

In [7]:


# ==========================================
# 1. The Core Router (Your provided class)
# ==========================================
class ImprovedContextAwareRouter(nn.Module):
    """Enhanced router with vocabulary-aware initialization and better scaling."""
    def __init__(
        self,
        vocab_size,
        num_experts, 
        router_dim,
        full_embed_dim,
        context_window=8,
        causal=True,
        num_heads=4,
        num_queries=8,
        n_layers=1,
        use_vocab_prior=True,
    ):
        super().__init__()
        # In this specific config, num_experts will equal target_dim (64)
        self.num_experts = num_experts 
        self.router_dim = router_dim
        self.context_window = context_window
        self.causal = causal
        self.num_heads = num_heads
        self.head_dim = router_dim // num_heads
        self.num_queries = num_queries
        self.n_layers = n_layers
        self.use_vocab_prior = use_vocab_prior
        
        # Project full embeddings to router dimension
        self.embed_proj = nn.Linear(full_embed_dim, router_dim)
        
        # Multi-head attention
        self.qkv_proj = nn.Linear(router_dim, 3 * router_dim, bias=False)
        self.out_proj = nn.Linear(router_dim, router_dim)
        
        # Expert projection with layer norm for stability
        self.ln = nn.LayerNorm(router_dim)
        self.routing_queries = nn.Parameter(torch.randn(num_queries, router_dim))
        self.temperature_predictor = nn.Linear(router_dim, 1)
        
        self.expert_proj = nn.Linear(router_dim, num_experts)
        self.cross_expert_proj = nn.Linear(router_dim, num_experts)
        self.alpha_gate = nn.Linear(router_dim, 1)
        
        # Vocabulary prior for better initialization
        if use_vocab_prior:
            self.vocab_routing_bias = nn.Embedding(vocab_size, num_experts)
            nn.init.normal_(self.vocab_routing_bias.weight, mean=0, std=0.02)
        
        # Better initialization for large vocab
        nn.init.normal_(self.expert_proj.weight, mean=0, std=0.02)
        nn.init.normal_(self.cross_expert_proj.weight, mean=0, std=0.02)
        
    def forward(self, full_embeds, input_ids=None):
        batch_size, seq_len, _ = full_embeds.shape
        
        # Project to router dimension
        x = self.embed_proj(full_embeds)
        
        # Multi-head context attention
        for _ in range(self.n_layers):
            x = self._multi_head_attention(x)
        
        # Layer norm before expert projection
        x = self.ln(x)
        
        # Get expert logits
        self_attn_logits = self.expert_proj(x)
        cross_attn_logits = self.cross_expert_proj(self._cross_attention(self.routing_queries, x))
        
        alpha = torch.sigmoid(self.alpha_gate(x))
        expert_logits = alpha * self_attn_logits + (1 - alpha) * cross_attn_logits
        
        # Add vocabulary prior if available
        if self.use_vocab_prior and input_ids is not None:
            vocab_bias = self.vocab_routing_bias(input_ids)
            expert_logits = expert_logits + vocab_bias
        
        adaptive_temp = torch.sigmoid(self.temperature_predictor(x)) * 2.0 + 0.1
        
        # We return expert_logits as the embedding itself
        return expert_logits, adaptive_temp, self.expert_proj.weight

    def _cross_attention(self, queries, context):
        batch_size, seq_len, dim = context.shape
        num_queries = queries.shape[0]
        
        queries = queries.unsqueeze(0).expand(batch_size, num_queries, dim)
        attn_scores = torch.matmul(queries, context.transpose(1, 2)) / (dim ** 0.5)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attended = torch.matmul(attn_weights, context)
        fused = attended.mean(dim=1).unsqueeze(1).expand(-1, seq_len, -1)
        return fused
    
    def _multi_head_attention(self, x):
        batch_size, seq_len, _ = x.shape
        
        qkv = self.qkv_proj(x).reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        mask = self._create_mask(seq_len, x.device).unsqueeze(0).unsqueeze(0)
        attn_scores = attn_scores.masked_fill(mask, float('-inf'))
        
        attn_weights = F.softmax(attn_scores, dim=-1)
        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).reshape(batch_size, seq_len, self.router_dim)
        output = self.out_proj(context) + x
        return output
    
    def _create_mask(self, seq_len, device):
        if self.context_window == -1:
            if self.causal:
                positions = torch.arange(seq_len, device=device)
                pos_diff = positions.unsqueeze(0) - positions.unsqueeze(1)
                return pos_diff > 0
            else:
                return torch.zeros(seq_len, seq_len, dtype=torch.bool, device=device)
        
        positions = torch.arange(seq_len, device=device)
        pos_diff = positions.unsqueeze(0) - positions.unsqueeze(1)
        
        if self.causal:
            return (pos_diff > 0) | (pos_diff < -self.context_window)
        else:
            window_half = self.context_window // 2
            return torch.abs(pos_diff) > window_half


# Re-using your provided ImprovedContextAwareRouter class
# (Assuming it is defined as in your snippet above)

class GlobalContextManager(nn.Module):
    """
    The 'Brain'. Runs once per forward pass.
    Extracts a compact context representation to control all downstream layers.
    """
    def __init__(self, vocab_size, d_model, router_dim=64, context_window=128):
        super().__init__()
        # We reuse your existing robust router
        # We set num_experts to 'router_dim' because we want a dense context vector
        # not necessarily sparse expert logits for this specific role.
        self.router = ImprovedContextAwareRouter(
            vocab_size=vocab_size,
            num_experts=router_dim, # Output a dense vector of this size
            router_dim=router_dim,
            full_embed_dim=d_model,
            context_window=context_window,
            causal=True,
            num_heads=4,
            n_layers=2
        )
        self.router_dim = router_dim

    def forward(self, x_embeds, input_ids=None):
        """
        Args:
            x_embeds: [Batch, Seq, D_model]
        Returns:
            context_state: [Batch, Seq, router_dim] 
                           This is the 'control signal' for all layers.
        """
        # We use the 'expert_logits' as our continuous context representation
        # rather than softmaxing them into probabilities, as we want dense info.
        context_logits, _, _ = self.router(x_embeds, input_ids)
        
        # Normalize for stability in downstream projections
        context_state = F.layer_norm(context_logits, context_logits.shape[-1:])
        return context_state

class RemixedLinear(nn.Module):
    def __init__(self, in_features, out_features, context_dim, basis_size=64, remixed_linear_kwargs = dict(use_basis_gate=True, use_output_gate=True, use_context=True)):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.basis_size = basis_size
        self.use_basis_gate = remixed_linear_kwargs['use_basis_gate']
        self.use_output_gate = remixed_linear_kwargs['use_output_gate']
        self.use_context = remixed_linear_kwargs['use_context']
        # self.last_gate_stats = {}
        
        # 1. STATIC COMPONENTS (The "Body")
        # A. The Basis Bank (Input -> Basis)
        self.basis = nn.Linear(in_features, basis_size, bias=False)
        # self.register_buffer("global_step", torch.zeros(1))
        # self.annealer = TemperatureAnnealer(
        #     start=1.5,
        #     end=0.3,        # do NOT go to zero
        #     total_steps=5_000,
        #     mode="cosine"
        # )
        
        # B. The Template Mixing Matrix (Basis -> Output)
        # This represents the "standard" way features are combined.
        # We learn this once. It is NOT generated.
        self.template_mixing = nn.Parameter(torch.randn(out_features, basis_size))
        # self.temperature = nn.Parameter(torch.tensor(1.0))
        # 2. DYNAMIC COMPONENTS (The "Controller")
        # instead of generating the whole matrix, we generate two small vectors:
        # a. Basis Gate: Which basis features are active? (Size: 64)
        # b. Output Gate: Which output neurons are active? (Size: 512)
        self.context_modulator = nn.Sequential(
            nn.Linear(context_dim, basis_size // 2),
            nn.GELU(),
            # Output size: basis_size + out_features (e.g., 64 + 512)
            nn.Linear(basis_size // 2, basis_size + out_features),
            # nn.Sigmoid() # Gating (0 to 1) or Tanh (-1 to 1) depending on preference
        )
        
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.ln_basis = nn.LayerNorm(basis_size)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.orthogonal_(self.basis.weight)
        nn.init.kaiming_normal_(self.template_mixing)
        # Initialize modulator to close to 1.0 (Identity pass-through)
        nn.init.constant_(self.context_modulator[-1].bias, 2.0) 

    def forward(self, x, context_state):
        B, S, C = x.shape
        # self.global_step += 1
    
        # tau = self.annealer(int(self.global_step.item()))
        # x: [Batch, Seq, In]
        # context_state: [Batch, Seq, Context_Dim]

        # 1. Project to Basis
        h_basis = self.ln_basis(self.basis(x)) # [B, S, Basis]
        
        # 2. Get Dynamic Modulation Gates
        # Shape: [B, S, Basis + Out]
        if self.use_context:
            gates = torch.sigmoid(self.context_modulator(context_state))#/self.temperature)
            B, S, C = gates.shape
            # Split into Basis Gate and Output Gate
            
            
            if not self.use_basis_gate:
                gate_basis = torch.ones(B, S, self.basis_size, device=device)
            else:
                gate_basis = gates[..., :self.basis_size]   # [B, S, Basis]
            if not self.use_output_gate:
                gate_out = torch.ones(B, S, C - self.basis_size, device=device)
            else:
                gate_out = gates[..., self.basis_size:]     # [B, S, Out]
            h_gated = h_basis * gate_basis
            
            # B. Apply Static Template Mixing (The heavy lifting matrix mul)
            # [B, S, Basis] @ [Basis, Out] -> [B, S, Out]
            pre_output = F.linear(h_gated, self.template_mixing)
        else:
            gate_basis = torch.ones_like(h_basis)
            h_gated = h_basis * gate_basis
            # B. Apply Static Template Mixing (The heavy lifting matrix mul)
            # [B, S, Basis] @ [Basis, Out] -> [B, S, Out]
            pre_output = F.linear(h_gated, self.template_mixing)
            gate_out   = torch.ones_like(pre_output)

        # if not self.use_basis_gate:
        #     gate_basis = torch.ones_like(gate_basis)

        # if not self.use_output_gate:
        #     gate_out = torch.ones_like(gate_out)

        self.last_gate_entropy = gate_entropy(gate_out)

        # Check the spread of your Basis Gate
        # We want to see a standard deviation > 0.
        # gate_std = gates[..., :self.basis_size].std()
        # gate_mean = gates[..., :self.basis_size].mean()
        # if self.global_step % 1000 ==0:
        #     print(f"Gate Mean: {gate_mean:.4f} | Gate Std: {gate_std:.4f}")

        # if self.training:
        #     self.last_gate_stats = {
        #         "basis": gate_stats(gate_basis.detach()),
        #         "out":   gate_stats(gate_out.detach())
        #     }
        
        # 3. The "Modulated" Remix
        # We want to perform: Output = (Basis * Gate_Basis) @ Template.T * Gate_Out
        
        # A. Apply Basis Gating (Context decides which input features matter)

        
        # C. Apply Output Gating (Context decides which output neurons fire)
        output = pre_output * gate_out
        
        return output + self.bias


class DynamicRemixNet(nn.Module):
    """Example Model putting it all together"""
    def __init__(self, vocab_size, d_model, n_layers, context_dim=64):
        super().__init__()
        
        self.embeddings = nn.Embedding(vocab_size, d_model)
        
        # 1. The Global Brain
        self.context_manager = GlobalContextManager(vocab_size, d_model, router_dim=context_dim)
        
        # 2. The Layers (using RemixedLinear instead of nn.Linear)
        self.layers = nn.ModuleList([
            RemixedLinear(d_model, d_model, context_dim, basis_size=64)
            for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        x = self.embeddings(input_ids)
        
        # 1. Compute Context ONCE
        context_state = self.context_manager(x, input_ids)
        
        # 2. Pass context to all dynamic layers
        for layer in self.layers:
            # Standard residual connection
            residual = x
            x = layer(x, context_state)
            x = self.norm(x + residual)
            
        return self.head(x)

# --- Verification Code ---
if __name__ == "__main__":
    B, S, D = 2, 64, 512
    V = 50257
    
    model = DynamicRemixNet(vocab_size=V, d_model=D//8, n_layers=4)
    inputs = torch.randint(0, V, (B, S))
    
    # Forward pass
    out = model(inputs)
    
    print(f"Input: {inputs.shape}")
    print(f"Output: {out.shape}")
    
    # Parameter Efficiency Check
    std_linear = nn.Linear(D,D) #D * D
    remix_linear = RemixedLinear(D//8, D//8, 64, basis_size=64) #(D * 64) + (64 * 64 * D) # Basis + Hypernet approximation
    # Note: Hypernet can be optimized further with low-rank factorization
    
    print(f"\nStandard Linear Layer Params: {sum(p.numel() for p in std_linear.parameters()):,}")
    print(f"Remixed Linear Layer Params:  {sum(p.numel() for p in remix_linear.parameters()):,}")

    # print(f"Remixed Linear Layer Params:  {sum(p.numel() for p in model.layers[0].parameters()):,}")
    print("\nConcept validated: Global Context drives Local Remixing.")

Input: torch.Size([2, 64])
Output: torch.Size([2, 64, 50257])

Standard Linear Layer Params: 262,656
Remixed Linear Layer Params:  14,688

Concept validated: Global Context drives Local Remixing.


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

# ==============================================================================
# 2. TRANSFORMER COMPONENTS (Modified for Remixing)
# ==============================================================================
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)

        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

# class MultiHeadAttention(nn.Module):
#     """ multiple heads of self-attention in parallel """

#     def __init__(self, n_embd, num_heads, head_size):
#         super().__init__()
#         self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
#         self.proj = nn.Linear(n_embd, n_embd)
#         self.dropout = nn.Dropout(dropout)

#     def forward(self, x):
#         out = torch.cat([h(x) for h in self.heads], dim=-1)
#         out = self.dropout(self.proj(out))
#         return out
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_heads, head_size, dropout, block_size):
        super().__init__()
        self.n_heads = n_heads
        self.head_size = head_size
        self.dropout_p = dropout
        
        # Projections for Q, K, V
        self.block_proj = nn.Linear(n_embd, n_embd * 3)
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(n_embd, n_embd)
        
        # Causal mask
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    
    def forward(self, blocks, attention_mask=None):
        B, T, C = blocks.shape
        
        # 1. Project to Q, K, V
        Q, K, V = self.block_proj(blocks).chunk(3, dim=-1)
        
        # 2. Split into heads: (B, T, C) -> (B, n_heads, T, head_size)
        Q = Q.view(B, T, self.n_heads, self.head_size).permute(0, 2, 1, 3)
        K = K.view(B, T, self.n_heads, self.head_size).permute(0, 2, 1, 3)
        V = V.view(B, T, self.n_heads, self.head_size).permute(0, 2, 1, 3)
        
        # 3. Apply attention with masking
        if attention_mask is not None:
            # Causal mask: (T, T)
            causal_mask = self.tril[:T, :T]
            
            # Padding mask: attention_mask is (B, T) where 1=valid, 0=padding
            # Create (B, T, T) mask where both query and key must be valid
            padding_mask = attention_mask.unsqueeze(1) * attention_mask.unsqueeze(2)  # (B, T, T)
            
            # Combine masks: both causal AND padding must allow attention
            combined_mask = causal_mask.unsqueeze(0) & padding_mask.bool()  # (B, T, T)
            combined_mask = combined_mask.unsqueeze(1)  # (B, 1, T, T) for broadcasting
            
            output = F.scaled_dot_product_attention(
                Q, K, V,
                attn_mask=combined_mask,
                dropout_p=self.dropout_p if self.training else 0.0
            )
        else:
            # Only causal mask (use is_causal for efficiency)
            output = F.scaled_dot_product_attention(
                Q, K, V,
                is_causal=True,
                dropout_p=self.dropout_p if self.training else 0.0
            )
        
        # 4. Concatenate heads and project
        output = output.transpose(1, 2).contiguous().view(B, T, C)
        output = self.dropout(self.proj(output))
        
        return output




class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head, dropout, block_size):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_embd, n_head, head_size, dropout, block_size)
        # self.sa = MultiTokenAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd,dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
class RemixedFeedFoward(nn.Module):
    """
    FeedForward Network using RemixedLinear.
    Replaces static up/down projections with dynamic, context-aware remixing.
    """
    def __init__(self, n_embd, dropout, context_dim, basis_size=64, remixed_linear_kwargs = dict(use_basis_gate=True, use_output_gate=True, use_context=True)):
        super().__init__()
        # 1. Expansion: n_embd -> 4 * n_embd
        # Uses dynamic mixing to expand features
        self.w1 = RemixedLinear(
            in_features=n_embd, 
            out_features=4 * n_embd, 
            context_dim=context_dim,
            basis_size=basis_size, remixed_linear_kwargs = remixed_linear_kwargs
        )
        
        self.act = nn.ReLU()
        
        # 2. Projection: 4 * n_embd -> n_embd
        # Uses dynamic mixing to compress back to model dim
        self.w2 = RemixedLinear(
            in_features=4 * n_embd, 
            out_features=n_embd, 
            context_dim=context_dim,
            basis_size=basis_size, remixed_linear_kwargs=remixed_linear_kwargs
        )
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, context_state):
        # Pass context_state to both RemixedLinear layers
        x = self.w1(x, context_state)
        x = self.act(x)
        x = self.w2(x, context_state)
        x = self.dropout(x)
        return x

class RemixedMultiHeadAttention(nn.Module):
    """Attention mechanism using RemixedLinear for Q, K, V projections."""
    def __init__(self, n_embd, n_heads, head_size, dropout, block_size, context_dim, basis_size=64, remixed_linear_kwargs = dict(use_basis_gate=True, use_output_gate=True, use_context=True)):
        super().__init__()
        self.n_heads = n_heads
        self.head_size = head_size
        self.dropout_p = dropout
        
        # REPLACE Standard Linear with RemixedLinear for QKV
        self.qkv_proj = RemixedLinear(
            in_features=n_embd, 
            out_features=n_embd * 3, 
            context_dim=context_dim,
            basis_size=basis_size,remixed_linear_kwargs = remixed_linear_kwargs 
        )
        
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(n_embd, n_embd) # Keeping output proj standard for aggregation
        
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    
    def forward(self, blocks, context_state, attention_mask=None):
        B, T, C = blocks.shape
        
        # 1. Remixed Projection
        qkv = self.qkv_proj(blocks, context_state)
        Q, K, V = qkv.chunk(3, dim=-1)
        
        # 2. Split Heads
        Q = Q.view(B, T, self.n_heads, self.head_size).permute(0, 2, 1, 3)
        K = K.view(B, T, self.n_heads, self.head_size).permute(0, 2, 1, 3)
        V = V.view(B, T, self.n_heads, self.head_size).permute(0, 2, 1, 3)
        
        # 3. Attention Logic
        if attention_mask is not None:
            causal_mask = self.tril[:T, :T]
            padding_mask = attention_mask.unsqueeze(1) * attention_mask.unsqueeze(2)
            combined_mask = causal_mask.unsqueeze(0) & padding_mask.bool()
            combined_mask = combined_mask.unsqueeze(1)
            
            output = F.scaled_dot_product_attention(
                Q, K, V, attn_mask=combined_mask, 
                dropout_p=self.dropout_p if self.training else 0.0
            )
        else:
            output = F.scaled_dot_product_attention(
                Q, K, V, is_causal=True, 
                dropout_p=self.dropout_p if self.training else 0.0
            )
        
        output = output.transpose(1, 2).contiguous().view(B, T, C)
        output = self.dropout(self.proj(output))
        return output

class RemixedBlock(nn.Module):
    """Transformer Block that accepts context_state"""
    def __init__(self, n_embd, n_head, dropout, block_size, context_dim, basis_size=64, remixed_linear_kwargs = dict(use_basis_gate=True, use_output_gate=True, use_context=True)):
        super().__init__()
        head_size = n_embd // n_head
        # Pass basis_size to Attention as well (optional consistency)
        self.sa = RemixedMultiHeadAttention(
            n_embd, n_head, head_size, dropout, block_size, context_dim, basis_size, remixed_linear_kwargs = remixed_linear_kwargs
        )
        # CHANGE: Initialize FeedFoward with context_dim and basis_size
        self.ffwd = RemixedFeedFoward(n_embd, dropout, context_dim, basis_size, remixed_linear_kwargs = remixed_linear_kwargs)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, context_state):
        x = x + self.sa(self.ln1(x), context_state)
        # CHANGE: Pass context_state to FeedFoward
        x = x + self.ffwd(self.ln2(x), context_state)
        return x

# ==============================================================================
# 3. THE MAIN MODEL
# ==============================================================================

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, seq_len, **kwargs):
        super().__init__()
        
        # --- Parsing Kwargs ---
        allowed_keys = {
            "use_moe", "num_experts", 'total_embed_dim', "router_dim", "capacity_factor",
            "use_sparse_top_k", "top_k", "routing_mode", "context_window",
            "causal", "use_expert_mlp", "use_output_projection",
            "use_expert_bias", "dropout", "use_shared_base", "shared_base_dim", 
            "use_vocab_prior", "expert_residual", 'allow_replacement', 
            'use_embed_refine', 'target_dim', 'selection_mode', "use_perm",
            
            # NEW KWARGS FOR LINEAR REMIXING
            "use_remixed_linear", "context_dim", "linear_basis_size", "remixed_linear_kwargs"
        }

        for key in kwargs:
            if key not in allowed_keys:
                print(f"Warning: Unexpected keyword argument '{key}'")

        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.block_size = seq_len # Consistency alias
        self.entropy_weight = 1e-3


        # Embedding & MoE Config
        self.use_moe = kwargs.get("use_moe", False)
        self.use_perm = kwargs.get("use_perm", False)
        self.use_embed_refine = kwargs.get("use_embed_refine", False)
        self.remixed_linear_kwargs = kwargs.get("remixed_linear_kwargs", dict(use_basis_gate=True, use_output_gate=True, use_context=True))
        self.num_experts = kwargs.get("num_experts", 8)
        self.router_dim = kwargs.get("router_dim", 64)
        self.target_dim = kwargs.get("target_dim", 64)
        self.total_embed_dim = kwargs.get("total_embed_dim", self.num_experts*self.router_dim)
        self.selection_mode = kwargs.get("selection_mode", "soft")
        self.allow_replacement = kwargs.get("allow_replacement", False)
        self.dropout = kwargs.get("dropout", 0.0)
        
        # Linear Remixing Config
        self.use_remixed_linear = kwargs.get("use_remixed_linear", False)
        self.context_dim = kwargs.get("context_dim", 64)
        self.linear_basis_size = kwargs.get("linear_basis_size", 64)

        # Calculate Model Dimension (n_embd)
        if self.use_moe:
            self.n_embd = self.target_dim
        else:
            self.n_embd = self.total_embed_dim
            


        # ----------------------------------------------------------------------
        # 1. Embedding Layers
        # ----------------------------------------------------------------------
        if self.use_moe:
            if self.use_perm:
                self.embedding_model = PermutationMoE(
                    vocab_size=vocab_size,
                    base_embed_dim=self.n_embd,
                    num_experts=self.num_experts,
                    selection_mode=self.selection_mode,
                    router_dim=self.router_dim,
                    allow_replacement=self.allow_replacement,
                    use_dim_mlp_refinement=True,
                    mlp_refinement_position='after',
                )
            else:
                self.embedding_model = DirectContextualEmbedding(
                    vocab_size=vocab_size, dim=self.n_embd
                )
        else:
            self.token_embedding_table = nn.Embedding(vocab_size, self.n_embd)
            self.position_embedding_table = nn.Embedding(self.block_size, self.n_embd)
            
            if self.use_embed_refine:
                self.refine_embed_layer = nn.Sequential(
                    nn.LayerNorm(self.n_embd),
                    nn.Linear(self.n_embd, self.n_embd * 2),
                    nn.GELU(),
                    nn.Dropout(self.dropout),
                    nn.Linear(self.n_embd * 2, self.n_embd)
                )
            self.embd_ln = nn.LayerNorm(self.n_embd)
            

        # ----------------------------------------------------------------------
        # 2. Global Context Manager (The "Brain")
        # ----------------------------------------------------------------------
        if self.use_remixed_linear:
            self.context_manager = GlobalContextManager(
                vocab_size=vocab_size, 
                d_model=self.n_embd, 
                router_dim=self.context_dim,
                context_window=self.block_size
            )

        # ----------------------------------------------------------------------
        # 3. Transformer Blocks (The "Body")
        # ----------------------------------------------------------------------
        # ... inside BigramLanguageModel.__init__ ...
        if self.use_remixed_linear:
            # Use ModuleList for Remixed Blocks
            self.blocks = nn.ModuleList([
                RemixedBlock(
                    self.n_embd, n_head, self.dropout, 
                    self.block_size, self.context_dim,
                    basis_size=self.linear_basis_size, remixed_linear_kwargs = self.remixed_linear_kwargs # <--- ADD THIS LINE
                ) 
                for _ in range(n_layer)
            ])
        else:
            # Standard Sequential Blocks
            self.blocks = nn.Sequential(*[
                Block(self.n_embd, n_head, self.dropout, self.block_size) 
                for _ in range(n_layer)
            ])

        # ----------------------------------------------------------------------
        # 4. Heads
        # ----------------------------------------------------------------------
        self.ln_f = nn.LayerNorm(self.n_embd)
        self.lm_head = nn.Linear(self.n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        device = idx.device
        routing_info = None

        # --- A. Embeddings ---
        if self.use_moe:
            # Context-Aware Embedding Routing
            if self.use_perm:
                tok_emb, routing_info = self.embedding_model(idx) 
            else:
                tok_emb = self.embedding_model(idx)
            x = tok_emb
        else:
            # Standard Static Embeddings
            tok_emb = self.token_embedding_table(idx)
            pos_emb = self.position_embedding_table(torch.arange(T, device=device))
            x = tok_emb + pos_emb
            
            if self.use_embed_refine:
                x = self.refine_embed_layer(x)
            if not self.use_remixed_linear:
                x = self.embd_ln(x)

        # --- B. Global Context Calculation ---
        context_state = None
        if self.use_remixed_linear:
            # Run the router once to get the global control signal
            # Note: We pass the embeddings 'x' to the router
            context_state = self.context_manager(x, idx)

        # --- C. Transformer Blocks ---
        if self.use_remixed_linear:
            # Manual loop to pass context_state
            for block in self.blocks:
                x = block(x, context_state)
        else:
            # Standard sequential execution
            x = self.blocks(x)

        # --- D. Output ---
        x = self.ln_f(x)
        logits = self.lm_head(x)

        # --- E. Loss ---
        loss = None
        total_loss = None
        
        if targets is not None:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
            
            if self.use_moe and routing_info and 'total_aux_loss' in routing_info:
                total_loss = loss #+ routing_info['total_aux_loss']
            else:
                total_loss = loss
        if self.use_remixed_linear:
            gate_entropy = collect_gate_entropy(self)
            if gate_entropy is not None:
                total_loss = loss + self.entropy_weight * gate_entropy
        return loss, total_loss, routing_info

    def generate(self, idx, max_new_tokens):
        for _ in tqdm(range(max_new_tokens), desc="Generating"):
            # Crop context if too long
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            
            # Forward pass
            _, _, _ = self(idx_cond) # We need to get logits, but forward returns losses
            # Wait, forward returns (loss, total_loss, routing). We need LOGITS.
            # FIX: We need to access the logic inside forward that produces logits.
            # OR modify forward to return logits if targets is None.
            
            # Re-running forward logic locally for generation (cleaner than refactoring return signature significantly)
            # NOTE: For cleaner code, one usually splits forward() into body() and loss(), 
            # but per your request to recreate the class, I will patch it here efficiently.
            
            # ... (Embedding) ...
            B, T = idx_cond.shape
            device = idx_cond.device
            
            if self.use_moe:
                 if self.use_perm: tok_emb, _ = self.embedding_model(idx_cond)
                 else: tok_emb = self.embedding_model(idx_cond)
                 x = tok_emb
            else:
                tok_emb = self.token_embedding_table(idx_cond)
                pos_emb = self.position_embedding_table(torch.arange(T, device=device))
                x = tok_emb + pos_emb
                if self.use_embed_refine: x = self.refine_embed_layer(x)
                x = self.embd_ln(x)
            
            # ... (Context) ...
            context_state = None
            if self.use_remixed_linear:
                context_state = self.context_manager(x, idx_cond)
                
            # ... (Blocks) ...
            if self.use_remixed_linear:
                for block in self.blocks: x = block(x, context_state)
            else:
                x = self.blocks(x)
                
            x = self.ln_f(x)
            logits = self.lm_head(x)
            
            
            # ... (Sampling) ...
            logits = logits[:, -1, :] 
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx

In [9]:


class TextDataset(Dataset):
    """Simple text dataset that returns sequences of tokens"""
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
    
    def __len__(self):
        return max(1, (len(self.data) - 1) // self.seq_len)
    
    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len
        x = self.data[start:end]
        y = self.data[start+1:end+1]
        
        # Pad if necessary
        if len(x) < self.seq_len:
            x = np.pad(x, (0, self.seq_len - len(x)), constant_values=0)
            y = np.pad(y, (0, self.seq_len - len(y)), constant_values=0)
        
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


# class Tokenizer:
#     """Simple character or word-level tokenizer"""
#     def __init__(self, vocab_type='char'):
#         self.vocab_type = vocab_type
#         self.char_to_idx = {}
#         self.idx_to_char = {}
#         self.vocab_size = 0
    
#     def build_vocab(self, text: str, min_freq: int = 1):
#         """Build vocabulary from text"""
#         if self.vocab_type == 'char':
#             chars = sorted(set(text))
#             self.char_to_idx = {ch: i for i, ch in enumerate(chars)}
#             self.idx_to_char = {i: ch for i, ch in enumerate(chars)}
#             self.vocab_size = len(chars)
#         else:  # word-level
#             words = text.split()
#             counter = Counter(words)
#             vocab = [w for w, c in counter.items() if c >= min_freq]
#             vocab = ['<pad>', '<unk>'] + sorted(vocab)
#             self.char_to_idx = {w: i for i, w in enumerate(vocab)}
#             self.idx_to_char = {i: w for i, w in enumerate(vocab)}
#             self.vocab_size = len(vocab)
    
#     def encode(self, text: str) -> List[int]:
#         """Encode text to token indices"""
#         if self.vocab_type == 'char':
#             return [self.char_to_idx.get(ch, 0) for ch in text]
#         else:  # word-level
#             words = text.split()
#             unk_idx = self.char_to_idx.get('<unk>', 0)
#             return [self.char_to_idx.get(w, unk_idx) for w in words]
    
#     def decode(self, indices: List[int]) -> str:
#         """Decode token indices to text"""
#         if self.vocab_type == 'char':
#             return ''.join([self.idx_to_char.get(i, '') for i in indices])
#         else:  # word-level
#             return ' '.join([self.idx_to_char.get(i, '<unk>') for i in indices])


def download_file(url: str, filepath: str):
    """Download file from URL"""
    if os.path.exists(filepath):
        return
    
    print(f"Downloading {url}...")
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(filepath, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded to {filepath}")


def load_wikitext103(seq_len,seed) -> Tuple[str, str, str]:
    """Load WikiText-103 dataset"""
# Create datasets
    data_dir = "/kaggle/input/wikitext103-encoded/"
    train_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'train.bin'),
        block_size=seq_len, seed=seed
        # length=10000  # Number of samples per epoch (adjust as needed)
    )
    
    val_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'val.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    test_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'test.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    # base_url = "https://s3.amazonaws.com/research.metamind.io/wikitext"
    # data_dir = "data/wikitext-103"
    
    # # Download if needed
    # for split in ['train', 'valid', 'test']:
    #     url = f"{base_url}/wikitext-103-raw-v1.zip"
    #     zip_path = os.path.join(data_dir, "wikitext-103-raw-v1.zip")
        
    #     if not os.path.exists(zip_path):
    #         download_file(url, zip_path)
            
    #         # Extract
    #         with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    #             zip_ref.extractall(data_dir)
    
    # # Load splits
    # base_path = os.path.join(data_dir, "wikitext-103-raw")
    # with open(f"{base_path}/wiki.train.raw", 'r', encoding='utf-8') as f:
    #     train_text = f.read()
    # with open(f"{base_path}/wiki.valid.raw", 'r', encoding='utf-8') as f:
    #     val_text = f.read()
    # with open(f"{base_path}/wiki.test.raw", 'r', encoding='utf-8') as f:
    #     test_text = f.read()
    
    return train_dataset, val_dataset, test_dataset

# class WikiText103Dataset(IterableDataset):
#     def __init__(self, data_path, block_size, length=None):
#         """
#         Args:
#             data_path: Path to the .bin file
#             block_size: Sequence length for each sample
#             length: Optional dataset length (number of samples per epoch)
#                    If None, uses the maximum possible samples from the file
#         """
#         self.data_path = data_path
#         self.block_size = block_size
        
#         # Get file size to determine max samples
#         file_size = os.path.getsize(data_path)
#         num_tokens = file_size // np.dtype(np.uint16).itemsize
#         max_samples = num_tokens - block_size
        
#         self.length = length if length is not None else max_samples
#         self.max_samples = max_samples
    
#     def __iter__(self):
#         # Create memmap for this worker
#         data = np.memmap(self.data_path, dtype=np.uint16, mode='r')
        
#         # Handle multi-worker data loading
#         worker_info = torch.utils.data.get_worker_info()
#         if worker_info is not None:
#             # Split workload among workers
#             per_worker = int(np.ceil(self.length / worker_info.num_workers))
#             worker_id = worker_info.id
#             iter_start = worker_id * per_worker
#             iter_end = min(iter_start + per_worker, self.length)
#         else:
#             iter_start = 0
#             iter_end = self.length
        
#         # Generate samples
#         for _ in range(iter_end - iter_start):
#             # Random index for this sample
#             idx = np.random.randint(0, self.max_samples)
            
#             # Extract x and y
#             x = torch.from_numpy(data[idx:idx + self.block_size].astype(np.int64))
#             y = torch.from_numpy(data[idx + 1:idx + 1 + self.block_size].astype(np.int64))
            
#             yield x, y
    
#     def __len__(self):
#         return self.length


class WikiText103Dataset(IterableDataset):
    def __init__(self, data_path, block_size, length=None, seed=None):
        """
        Args:
            data_path: Path to the .bin file
            block_size: Sequence length for each sample
            length: Optional dataset length (number of samples per epoch)
            seed: Optional random seed for reproducibility
        """
        self.data_path = data_path
        self.block_size = block_size
        self.seed = seed

        file_size = os.path.getsize(data_path)
        num_tokens = file_size // np.dtype(np.uint16).itemsize
        max_samples = num_tokens - block_size

        self.length = length if length is not None else max_samples
        self.max_samples = max_samples

    def __iter__(self):
        # Create memmap for this worker
        data = np.memmap(self.data_path, dtype=np.uint16, mode='r')

        # Handle multi-worker data loading
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None:
            per_worker = int(np.ceil(self.length / worker_info.num_workers))
            worker_id = worker_info.id
            iter_start = worker_id * per_worker
            iter_end = min(iter_start + per_worker, self.length)

            # Adjust seed per worker for reproducibility
            seed = self.seed + worker_id if self.seed is not None else None
        else:
            iter_start = 0
            iter_end = self.length
            seed = self.seed

        # Set seed for reproducibility
        if seed is not None:
            np.random.seed(seed)

        # Generate samples
        for _ in range(iter_end - iter_start):
            idx = np.random.randint(0, self.max_samples)
            x = torch.from_numpy(data[idx:idx + self.block_size].astype(np.int64))
            y = torch.from_numpy(data[idx + 1:idx + 1 + self.block_size].astype(np.int64))
            yield x, y

    def __len__(self):
        return self.length



# Usage example

device ='cuda' if torch.cuda.is_available() else 'cpu'



def load_penn_treebank(seq_len,seed) -> Tuple[str, str, str]:
    """Load Penn Treebank dataset"""
    # Create datasets
    data_dir = "/kaggle/input/penn-treebank-encoded/"
    train_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'train.bin'),
        block_size=seq_len, seed=seed
        # length=10000  # Number of samples per epoch (adjust as needed)
    )
    
    val_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'val.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    test_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'test.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    # base_url = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data"
    # data_dir = "data/penn_treebank"
    
    # files = {
    #     'train': 'ptb.train.txt',
    #     'valid': 'ptb.valid.txt', 
    #     'test': 'ptb.test.txt'
    # }
    
    # texts = []
    # for split, filename in files.items():
    #     filepath = os.path.join(data_dir, filename)
    #     if not os.path.exists(filepath):
    #         url = f"{base_url}/{filename}"
    #         download_file(url, filepath)
        
    #     with open(filepath, 'r', encoding='utf-8') as f:
    #         texts.append(f.read())
    
    # return texts[0], texts[1], texts[2]
    return train_dataset, val_dataset, test_dataset

def load_enwik8(seq_len,seed) -> Tuple[str, str, str]:
    """Load enwik8 dataset (first 100M bytes of Wikipedia)"""
    data_dir = "/kaggle/input/enwik8-encoded/"
    train_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'train.bin'),
        block_size=seq_len, seed=seed
        # length=10000  # Number of samples per epoch (adjust as needed)
    )
    
    val_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'val.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    test_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'test.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    # url = "http://mattmahoney.net/dc/enwik8.zip"
    # data_dir = "data/enwik8"
    # zip_path = os.path.join(data_dir, "enwik8.zip")
    # file_path = os.path.join(data_dir, "enwik8")
    
    # if not os.path.exists(file_path):
    #     download_file(url, zip_path)
    #     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    #         zip_ref.extractall(data_dir)
    
    # with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
    #     text = f.read()
    
    # # Standard split: 90M train, 5M val, 5M test
    # n = len(text)
    # train_text = text[:90_000_000]
    # val_text = text[90_000_000:95_000_000]
    # test_text = text[95_000_000:100_000_000]
    
    # return train_text, val_text, test_text
    return train_dataset, val_dataset, test_dataset

def load_text8(seq_len,seed) -> Tuple[str, str, str]:
    """Load text8 dataset (cleaned enwik8)"""
    data_dir = "/kaggle/input/text8-encoded/"
    train_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'train.bin'),
        block_size=seq_len, seed=seed
        # length=10000  # Number of samples per epoch (adjust as needed)
    )
    
    val_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'val.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    test_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'test.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    # url = "http://mattmahoney.net/dc/text8.zip"
    # data_dir = "data/text8"
    # zip_path = os.path.join(data_dir, "text8.zip")
    # file_path = os.path.join(data_dir, "text8")
    
    # if not os.path.exists(file_path):
    #     download_file(url, zip_path)
    #     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    #         zip_ref.extractall(data_dir)
    
    # with open(file_path, 'r', encoding='utf-8') as f:
    #     text = f.read()
    
    # # Standard split: 90M train, 5M val, 5M test
    # n = len(text)
    # train_text = text[:90_000_000]
    # val_text = text[90_000_000:95_000_000]
    # test_text = text[95_000_000:]
    
    # return train_text, val_text, test_text
    return train_dataset, val_dataset, test_dataset

def load_tinyshakespeare(seq_len,seed) -> Tuple[str, str, str]:
    """Load WikiText-103 dataset"""
# Create datasets
    data_dir = "/kaggle/input/tinyshakespeare-encoded/"
    train_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'train.bin'),
        block_size=seq_len, seed=seed
        # length=10000  # Number of samples per epoch (adjust as needed)
    )
    
    val_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'val.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    test_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'test.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    return train_dataset, val_dataset, test_dataset


    
def load_lambada(seq_len,seed) -> Tuple[str, str, str]:
    """Load LAMBADA dataset"""
    data_dir = "/kaggle/input/lambada-encoded/"
    train_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'train.bin'),
        block_size=seq_len, seed=seed
        # length=10000  # Number of samples per epoch (adjust as needed)
    )
    
    val_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'val.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    test_dataset = WikiText103Dataset(
        data_path=os.path.join(data_dir, 'test.bin'),
        block_size=seq_len, seed=seed
        # length=1000  # Smaller for validation
    )
    # url = "https://raw.githubusercontent.com/cybertronai/bflm/master/lambada_test.jsonl"
    # data_dir = "data/lambada"
    # file_path = os.path.join(data_dir, "lambada_test.jsonl")
    
    # if not os.path.exists(file_path):
    #     download_file(url, file_path)
    
    # import json
    # texts = []
    # with open(file_path, 'r', encoding='utf-8') as f:
    #     for line in f:
    #         data = json.loads(line)
    #         texts.append(data['text'])
    
    # full_text = '\n'.join(texts)
    
    # # LAMBADA is test-only, so we'll create artificial splits
    # n = len(full_text)
    # train_text = full_text[:int(0.8*n)]
    # val_text = full_text[int(0.8*n):int(0.9*n)]
    # test_text = full_text[int(0.9*n):]
    
    # return train_text, val_text, test_text
    return train_dataset, val_dataset, test_dataset


def load_dataset(name: str, config,seed):
    """Load and prepare dataset with proper tokenization and data loaders
    
    Args:
        name: Dataset name
        config: Configuration object with attributes:
            - batch_size: Batch size for data loaders
            - seq_len: Sequence length for training
            - num_workers: Number of workers for data loading (default: 0)
    
    Returns:
        Tuple of (train_loader, val_loader, test_loader, vocab_size)
    """
    # Load dataset based on name
    dataset_loaders = {
        'wikitext-103': load_wikitext103,
        'penn_treebank': load_penn_treebank,
        'enwik8': load_enwik8,
        'text8': load_text8,
        'lambada': load_lambada,
        'tinyshakespeare' : load_tinyshakespeare,

    }
    
    if name not in dataset_loaders:
        raise ValueError(f"Unknown dataset: {name}. Available: {list(dataset_loaders.keys())}")
    
    print(f"\nLoading {name}...")
    # Determine tokenization type (char-level for enwik8/text8, word-level for others)
    vocab_type = 'char' if name in ['enwik8', 'text8'] else 'word'
    # Build tokenizer
    print("Building vocabulary...")
    tokenizer = tiktoken.get_encoding("gpt2")#Tokenizer(vocab_type=vocab_type)
    batch_size = getattr(config, 'BATCH_SIZE', 32)

    if name not in ['wikitext-103', 'penn_treebank','enwik8','text8','lambada', 'tinyshakespeare']:
        train_text, val_text, test_text = dataset_loaders[name](config)
        

        # tokenizer.build_vocab(train_text, min_freq=1)
        
        print(f"Vocabulary size: {tokenizer.n_vocab}")
        
        # Encode datasets
        print("Encoding datasets...")
        train_data = np.array(tokenizer.encode(train_text), dtype=np.int16)
        val_data = np.array(tokenizer.encode(val_text), dtype=np.int16)
        test_data = np.array(tokenizer.encode(test_text), dtype=np.int16)
        
        print(f"Train tokens: {len(train_data):,}")
        print(f"Val tokens: {len(val_data):,}")
        print(f"Test tokens: {len(test_data):,}")
        
        # Create datasets
        seq_len = getattr(config, 'SEQ_LEN', 512)
        train_dataset = TextDataset(train_data, seq_len)
        val_dataset = TextDataset(val_data, seq_len)
        test_dataset = TextDataset(test_data, seq_len)
        shuffle= True
    else:
        seq_len = getattr(config, 'SEQ_LEN', 512)
        shuffle= False
        train_dataset, val_dataset, test_dataset = dataset_loaders[name](seq_len,seed)
    
    # Create data loaders
    num_workers = getattr(config, 'num_workers', 2)
    batch_size = getattr(config, 'batch_size', 32)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        worker_init_fn=lambda worker_id: np.random.seed(seed + worker_id),
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        worker_init_fn=lambda worker_id: np.random.seed(seed + worker_id),
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        worker_init_fn=lambda worker_id: np.random.seed(seed + worker_id),
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader, tokenizer.n_vocab




In [10]:
"""
Mixture-of-Experts Embeddings: Comprehensive Evaluation Framework
=================================================================

This framework evaluates MoE embedding approaches across multiple datasets
and compares against baselines for research paper preparation.

Key Contributions:
1. Sparse activation (64-dim) outperforms dense (512-dim)
2. Soft routing via weighted expert combinations
3. Context-aware expert selection
4. Optional: Free dimension selector with dynamic expert ordering
"""

n_layer = 6 
n_head = 8


    

# ============================================================================
# METRICS TRACKING
# ============================================================================
def to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    elif hasattr(obj, 'tolist'):  # for tensors or arrays
        return obj.tolist()
    return obj


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer, np.int32, np.int64)):
            return int(obj)
        elif isinstance(obj, (np.floating, np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, defaultdict):
            return dict(obj)
        return super().default(obj)



"""
Mixture-of-Experts Embeddings: Multi-Seed Evaluation Framework
================================================================

Robust evaluation across multiple random seeds with statistical analysis.
"""

# import torch
# import torch.nn as nn
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from collections import defaultdict
# from typing import Dict, List, Tuple
# import json
# from tqdm.notebook import tqdm
# import matplotlib.pyplot as plt
# import seaborn as sns

# ============================================================================
# EVALUATION CONFIGURATION
# ============================================================================

class EvalConfig:
    """Configuration for systematic evaluation"""
    
    # Datasets to evaluate (add your specific datasets)
    DATASETS = [
        # 'tinyshakespeare',
        'wikitext-103',
        # 'penn_treebank', 
        # 'enwik8',
        # 'text8',
        # 'lambada',
    ]
    # Load dataset based on name
    DATA_DIR = {
        'wikitext-103': "/kaggle/input/wikitext103-encoded/",
        'penn_treebank': "/kaggle/input/penn-treebank-encoded/",
        'enwik8': "/kaggle/input/enwik8-encoded/",
        'text8': "/kaggle/input/text8-encoded/",
        'lambada': "/kaggle/input/lambada-encoded/",
        'tinyshakespeare' : "/kaggle/input/tinyshakespeare-encoded/",
    }
    
    
    MODEL_CONFIGS = {
        # 'baseline_512': {
        #     'use_moe': False,
        #     'total_embed_dim': 512,
        # },


        # 'baseline_64_no_refine_embd': {
        #     'use_moe': False,
        #     'use_embed_refine':False,
        #     'total_embed_dim': 64,
        # },

        # 'baseline_256': {
        #     'use_moe': False,
        #     'total_embed_dim': 256,
        # },
        'moe_remixed_lin': {
            'use_moe': False,
            'total_embed_dim': 64,
            'use_remixed_linear':True,
        },

        # 'moe_dense_8x64_no_replacement_soft': {
        #     'use_moe': True,
        #     'total_embed_dim': 512,
        #     'target_dim': 64,
        #     "use_perm": True,
        #     "use_remixed_linear":False,

        #     'use_shared_base': False,
        #     'shared_base_dim': 64,
        #     'expert_residual':False,  # Add residual connections in experts
        #     'use_vocab_prior':False,
        #     'use_expert_bias':False,
        #     'router_dim': 64,
        #     'allow_replacement' : False,
        #     'selection_mode': 'soft',

        #     'num_experts': 8,
        #     'use_sparse_top_k': False,
        #     'routing_mode': 'token_choice'
        # },
        # 'moe_direct_contextual': {
        #     'use_moe': True,
        #     'total_embed_dim': 512,
        #     'target_dim': 64,
        #     # "use_perm": False,

        #     'use_shared_base': False,
        #     'shared_base_dim': 64,
        #     'expert_residual':False,  # Add residual connections in experts
        #     'use_vocab_prior':False,
        #     'use_expert_bias':False,
        #     'router_dim': 64,
        #     'allow_replacement' : False,
        #     'selection_mode': 'soft',

        #     'num_experts': 8,
        #     'use_sparse_top_k': False,
        #     'routing_mode': 'token_choice'
        # },
        # 'moe_remixed_linear_direct_contextual': {
        #     'use_moe': True,
        #     'total_embed_dim': 64,
        #     'target_dim': 64,
        #     "use_remixed_linear": True,

        #     'use_shared_base': False,
        #     'shared_base_dim': 64,
        #     'expert_residual':False,  # Add residual connections in experts
        #     'use_vocab_prior':False,
        #     'use_expert_bias':False,
        #     'router_dim': 64,
        #     'allow_replacement' : False,
        #     'selection_mode': 'soft',

        #     'num_experts': 8,
        #     'use_sparse_top_k': False,
        #     'routing_mode': 'token_choice'
        # },
        # 'moe_remixed_linear_perm': {
        #     'use_moe': True,
        #     'total_embed_dim': 64,
        #     'target_dim': 64,
        #     "use_remixed_linear": True,
        #     "use_perm":True,

        #     'use_shared_base': False,
        #     'shared_base_dim': 64,
        #     'expert_residual':False,  # Add residual connections in experts
        #     'use_vocab_prior':False,
        #     'use_expert_bias':False,
        #     'router_dim': 64,
        #     'allow_replacement' : False,
        #     'selection_mode': 'soft',

        #     'num_experts': 8,
        #     'use_sparse_top_k': False,
        #     'routing_mode': 'token_choice'
        # },
    }

    
    # Training hyperparameters
    BATCH_SIZE = 32
    SEQ_LEN = 64
    MAX_ITERS = 5000
    EVAL_INTERVAL = 1000
    EVAL_ITERS = 200
    MOE_SCALE = 5
    LEARNING_RATE = 1e-3 #* MOE_SCALE
    WARMUP_PCT = 0.3
    PRINT_INTERVAL = 1
    
    # Multi-seed configuration
    NUM_SEEDS = 1  # Run 5 times for statistical significance
    BASE_SEED = 42
    SEEDS = [0,1,42,1337]


# ============================================================================
# ENHANCED METRICS TRACKER WITH MULTI-SEED SUPPORT
# ============================================================================

class MultiSeedMetricsTracker:
    """Track metrics across multiple seeds with statistical analysis"""
    
    def __init__(self, save_dir: str):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        # Structure: {dataset: {model: {seed: {metric: values}}}}
        self.metrics = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
        
    def log(self, dataset: str, model: str, seed: int, metric: str, value: float, step: int):
        """Log a metric for a specific seed"""
        key = f"{dataset}/{model}/{seed}"
        if metric not in self.metrics[dataset][model][seed]:
            self.metrics[dataset][model][seed][metric] = []
        self.metrics[dataset][model][seed][metric].append({
            'step': step,
            'value': value
        })
    
    def log_final(self, dataset: str, model: str, seed: int, metrics_dict: Dict):
        """Log final test metrics for a specific seed"""
        self.metrics[dataset][model][seed]['final'] = metrics_dict
    
    def save(self):
        """Save all metrics to disk"""
        # Convert defaultdict to regular dict for JSON serialization
        regular_dict = {
            dataset: {
                model: {
                    str(seed): seed_data
                    for seed, seed_data in model_data.items()
                }
                for model, model_data in dataset_data.items()
            }
            for dataset, dataset_data in self.metrics.items()
        }
        
        with open(self.save_dir / 'metrics_all_seeds.json', 'w') as f:
            json.dump(regular_dict, f, indent=2, cls=NumpyEncoder)
    
    def load(self):
        """Load metrics from disk"""
        metrics_file = self.save_dir / 'metrics_all_seeds.json'
        if not metrics_file.exists():
            raise FileNotFoundError(f"No metrics file found at {metrics_file}")
        
        with open(metrics_file, 'r') as f:
            loaded_data = json.load(f)
        
        # Convert back to defaultdict structure
        self.metrics = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
        for dataset, dataset_data in loaded_data.items():
            for model, model_data in dataset_data.items():
                for seed_str, seed_data in model_data.items():
                    seed = int(seed_str)
                    self.metrics[dataset][model][seed] = seed_data
        
        print(f"Loaded metrics from {metrics_file}")
        return self
    
    def get_aggregated_results(self) -> pd.DataFrame:
        """Aggregate results across seeds with mean and std"""
        results = []
        
        for dataset in self.metrics:
            for model in self.metrics[dataset]:
                # Collect final perplexities across all seeds
                ppls = []
                losses = []
                
                for seed in self.metrics[dataset][model]:
                    final = self.metrics[dataset][model][seed].get('final', {})
                    if 'perplexity' in final:
                        ppls.append(final['perplexity'])
                    if 'loss' in final:
                        losses.append(final['loss'])
                
                if ppls:
                    results.append({
                        'Dataset': dataset,
                        'Model': model,
                        'PPL_Mean': np.mean(ppls),
                        'PPL_Std': np.std(ppls),
                        'PPL_Min': np.min(ppls),
                        'PPL_Max': np.max(ppls),
                        'Loss_Mean': np.mean(losses) if losses else None,
                        'Loss_Std': np.std(losses) if losses else None,
                        'Num_Seeds': len(ppls)
                    })
        
        return pd.DataFrame(results)
    
    def get_comparison_table(self, format='pretty') -> str:
        """Generate comparison table with mean ± std"""
        df = self.get_aggregated_results()
        
        if df.empty:
            return "No results available"
        
        # Pivot for better display
        pivot_data = []
        for dataset in df['Dataset'].unique():
            row = {'Dataset': dataset}
            dataset_df = df[df['Dataset'] == dataset]
            
            for _, model_row in dataset_df.iterrows():
                model = model_row['Model']
                mean = model_row['PPL_Mean']
                std = model_row['PPL_Std']
                row[model] = f"{mean:.2f} ± {std:.2f}"
            
            pivot_data.append(row)
        
        pivot_df = pd.DataFrame(pivot_data).set_index('Dataset')
        
        if format == 'dataframe':
            return pivot_df
        elif format == 'markdown':
            return pivot_df.to_markdown()
        elif format == 'latex':
            return self._format_latex_table(df)
        else:
            return self._format_pretty_table(pivot_df)
    
    def _format_pretty_table(self, df: pd.DataFrame) -> str:
        """Format as pretty ASCII table with statistical markers"""
        lines = []
        
        # Column widths
        col_widths = {'Dataset': max(15, max(len(str(idx)) for idx in df.index))}
        for col in df.columns:
            col_widths[col] = max(len(col), 20)  # Space for "XX.XX ± X.XX"
        
        # Top border
        lines.append('┌' + '─' * (col_widths['Dataset'] + 2) + '┬' + 
                    '┬'.join('─' * (col_widths[col] + 2) for col in df.columns) + '┐')
        
        # Header
        header = '│ ' + 'Dataset'.ljust(col_widths['Dataset']) + ' │'
        for col in df.columns:
            header += ' ' + col.ljust(col_widths[col]) + ' │'
        lines.append(header)
        
        # Separator
        lines.append('├' + '─' * (col_widths['Dataset'] + 2) + '┼' + 
                    '┼'.join('─' * (col_widths[col] + 2) for col in df.columns) + '┤')
        
        # Data rows
        for idx, row in df.iterrows():
            line = '│ ' + str(idx).ljust(col_widths['Dataset']) + ' │'
            for col in df.columns:
                val = str(row[col]) if pd.notna(row[col]) else 'N/A'
                line += ' ' + val.ljust(col_widths[col]) + ' │'
            lines.append(line)
        
        # Bottom border
        lines.append('└' + '─' * (col_widths['Dataset'] + 2) + '┴' + 
                    '┴'.join('─' * (col_widths[col] + 2) for col in df.columns) + '┘')
        
        return '\n'.join(lines)
    
    def _format_latex_table(self, df: pd.DataFrame) -> str:
        """Generate LaTeX table with statistical formatting"""
        datasets = df['Dataset'].unique()
        models = df['Model'].unique()
        
        lines = [
            r"\begin{table}[t]",
            r"\centering",
            r"\caption{Perplexity comparison across datasets (mean $\pm$ std over 5 seeds)}",
            r"\label{tab:results}",
            r"\begin{tabular}{l" + "c" * len(models) + "}",
            r"\toprule",
            "Dataset & " + " & ".join(models) + r" \\",
            r"\midrule"
        ]
        
        for dataset in datasets:
            row = [dataset]
            dataset_df = df[df['Dataset'] == dataset]
            
            best_mean = dataset_df['PPL_Mean'].min()
            
            for model in models:
                model_data = dataset_df[dataset_df['Model'] == model]
                if not model_data.empty:
                    mean = model_data['PPL_Mean'].values[0]
                    std = model_data['PPL_Std'].values[0]
                    
                    cell = f"${mean:.2f} \\pm {std:.2f}$"
                    if mean == best_mean:
                        cell = r"\textbf{" + cell + "}"
                    row.append(cell)
                else:
                    row.append("--")
            
            lines.append(" & ".join(row) + r" \\")
        
        lines.extend([
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}"
        ])
        
        return "\n".join(lines)
    
    def perform_statistical_tests(self) -> pd.DataFrame:
        """Perform paired t-tests between models"""
        from scipy import stats
        
        results = []
        df = self.get_aggregated_results()
        
        for dataset in df['Dataset'].unique():
            dataset_df = df[df['Dataset'] == dataset]
            models = dataset_df['Model'].tolist()
            
            # Compare each pair of models
            for i, model1 in enumerate(models):
                for model2 in models[i+1:]:
                    # Get perplexities for each seed
                    ppls1 = []
                    ppls2 = []
                    
                    for seed in self.metrics[dataset][model1]:
                        final1 = self.metrics[dataset][model1][seed].get('final', {})
                        final2 = self.metrics[dataset][model2][seed].get('final', {})
                        
                        if 'perplexity' in final1 and 'perplexity' in final2:
                            ppls1.append(final1['perplexity'])
                            ppls2.append(final2['perplexity'])
                    
                    if len(ppls1) >= 3:  # Need at least 3 samples
                        t_stat, p_value = stats.ttest_rel(ppls1, ppls2)
                        
                        results.append({
                            'Dataset': dataset,
                            'Model_1': model1,
                            'Model_2': model2,
                            'Mean_Diff': np.mean(ppls1) - np.mean(ppls2),
                            'T_Statistic': t_stat,
                            'P_Value': p_value,
                            'Significant': p_value < 0.05
                        })
        
        return pd.DataFrame(results)

def get_optimizer(model, config, is_moe=False):
    if not is_moe:
        return torch.optim.AdamW(
            model.parameters(), 
            lr=config.LEARNING_RATE
        )
    
    # For MoE: split parameters
    expert_params = []
    non_expert_params = []
    
    for name, param in model.named_parameters():
        if any(x in name for x in ['embedding_model']): #'dim_selectors', 'expert_mlps', 'expert_mlp' 'embeddings', 'expert_router']):
            expert_params.append(param)
        else:
            non_expert_params.append(param)
    
    return torch.optim.AdamW([
        {'params': expert_params, 'lr': config.LEARNING_RATE * config.MOE_SCALE},
        {'params': non_expert_params, 'lr': config.LEARNING_RATE}
    ], weight_decay=0.01)
# ============================================================================
# TRAINING WITH SEED CONTROL
# ============================================================================

def set_seed(seed: int):
    """Set random seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    # Python random seed
    import random
    random.seed(seed)
    
    # Deterministic operations (may slow down training)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_and_evaluate_with_seed(
    model, train_loader, val_loader, test_loader,
    config, tracker, dataset_name, model_name, seed, device
):
    """Training pipeline with seed tracking"""
    # print(model.selection_mode)
    set_seed(seed)  # Reset seed before training
    # ADD THIS: Count and store parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    
    
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE *config.MOE_SCALE if "moe" in model_name else config.LEARNING_RATE,
        weight_decay=0.01
    )
    # optimizer = get_optimizer(model, config, is_moe="moe" in model_name)
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=config.LEARNING_RATE *config.MOE_SCALE if "moe" in model_name else config.LEARNING_RATE,
        total_steps=config.MAX_ITERS,
        pct_start=config.WARMUP_PCT
    )
    
    model.train()
    best_val_loss = float('inf')
    best_model_state = None
    train_iter = iter(train_loader)
    # grad_logger = GradientLogger()
    # grad_logger.attach(model)

    for step in tqdm(range(config.MAX_ITERS), desc=f"Seed {seed}"):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)
        
        x, y = x.to(device), y.to(device)
        
        # Forward pass
        loss, total_loss, info = model(x, y)
        
        # Backward pass
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        # Evaluation
        if step % config.EVAL_INTERVAL == 0 or step == config.MAX_ITERS - 1:
            val_results = evaluate_model(
                model, val_loader, device, config.EVAL_ITERS
            )
            
            # Log with seed
            tracker.log(dataset_name, model_name, seed, 'train_loss', loss.item(), step)
            tracker.log(dataset_name, model_name, seed, 'val_loss', val_results['loss'], step)
            tracker.log(dataset_name, model_name, seed, 'val_perplexity', val_results['perplexity'], step)
            # print("Context Modulator:", grad_logger.summary("context_modulator"))
            # print("Basis:", grad_logger.summary("basis"))
            # print("Template Mixing:", grad_logger.summary("template_mixing"))


            
            if step % (config.EVAL_INTERVAL * config.PRINT_INTERVAL) == 0:
                print(f"[Seed {seed}] Step {step}: val_ppl={val_results['perplexity']:.2f}")
            
            if val_results['loss'] < best_val_loss:
                best_val_loss = val_results['loss']
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            
            model.train()
    
    # Load best model and test
    if best_model_state:
        model.load_state_dict(best_model_state)
    
    test_results = evaluate_model(model, test_loader, device, config.EVAL_ITERS)
    test_results['total_params'] = total_params
    test_results['trainable_params'] = trainable_params
    tracker.log_final(dataset_name, model_name, seed, test_results)
    
    return test_results


def evaluate_model(model, dataloader, device, max_iters=100):
    """Evaluate model"""
    model.eval()
    losses = []
    
    with torch.no_grad():
        for i, (x, y) in enumerate(dataloader):
            if i >= max_iters:
                break
            
            x, y = x.to(device), y.to(device)
            loss, _, _ = model(x, y)
            losses.append(loss.item())
    
    model.train()
    
    avg_loss = np.mean(losses)
    return {
        'loss': avg_loss,
        'perplexity': np.exp(avg_loss)
    }


# ============================================================================
# MAIN EVALUATION WITH MULTI-SEED
# ============================================================================

def run_multi_seed_evaluation():
    """Run evaluation across multiple seeds"""
    
    config = EvalConfig()
    tracker = MultiSeedMetricsTracker('results')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("=" * 80)
    print(f"Multi-Seed Evaluation ({config.NUM_SEEDS} seeds)")
    print("=" * 80)
    
    for dataset_name in config.DATASETS:
        print(f"\n{'=' * 80}")
        print(f"Dataset: {dataset_name}")
        print(f"{'=' * 80}")
        

        
        for model_name, model_config in config.MODEL_CONFIGS.items():
            print(f"\n{'-' * 80}")
            print(f"Model: {model_name}")
            print(f"{'-' * 80}")
            
            # Run across multiple seeds
            for seed_idx in range(config.NUM_SEEDS):
                seed = config.BASE_SEED + seed_idx

            # for seed_idx, seed in enumerate(config.SEEDS):
                # num_seeds = len(config.SEEDS)
                # Load dataset
                train_loader, val_loader, test_loader, vocab_size = load_dataset(
                    dataset_name, config,seed)
                
        
                num_seeds = config.NUM_SEEDS
                print(f"\n  Seed {seed_idx + 1}/{num_seeds} (seed={seed})")
                
                # Build fresh model for each seed
                set_seed(seed)
                model = build_model(vocab_size, model_config, config).to(device)
                
                # Train and evaluate
                results = train_and_evaluate_with_seed(
                    model, train_loader, val_loader, test_loader,
                    config, tracker, dataset_name, model_name, seed, device
                )
                
                print(f"    Test PPL: {results['perplexity']:.2f}")
                torch.save(model.state_dict(), "model.pt")
                # Inside your training loop
                # total_norm_static = torch.norm(model.blocks[0].sa.qkv_proj.template_mixing.grad)
                # total_norm_dynamic = 0
                # for p in model.blocks[0].sa.qkv_proj.context_modulator.parameters():
                #     if p.grad is not None:
                #         total_norm_dynamic += torch.norm(p.grad)
                
                # print(f"Static Grad: {total_norm_static} | Dynamic Grad: {total_norm_dynamic}")
                # print(f"Gate Stats: {collect_gate_stats(model)}")
                # Clean up
                del model
                torch.cuda.empty_cache()
    
    # Save results
    tracker.save()
    
    # Display aggregated results
    print("\n" + "=" * 80)
    print("AGGREGATED RESULTS (Mean ± Std)")
    print("=" * 80 + "\n")
    
    print(tracker.get_comparison_table())
    
    # Statistical significance tests
    print("\n" + "=" * 80)
    print("STATISTICAL SIGNIFICANCE TESTS")
    print("=" * 80 + "\n")
    
    stats_df = tracker.perform_statistical_tests()
    print(stats_df.to_string())
    
    # Generate LaTeX table
    latex_path = tracker.save_dir / 'results_table.tex'
    with open(latex_path, 'w') as f:
        f.write(tracker.get_comparison_table(format='latex'))
    print(f"\nLaTeX table saved to: {latex_path}")
    # Generate plots
    plot_comparison_results(tracker, 'results/figures')
    
    print("\nEvaluation complete! Results saved to 'results/' directory")
    
    return tracker




# # ============================================================================
# # VISUALIZATION
# # ============================================================================


def plot_seed_variance(tracker: MultiSeedMetricsTracker, save_dir: str):
    """Plot variance across seeds"""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    df = tracker.get_aggregated_results()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for dataset in df['Dataset'].unique():
        dataset_df = df[df['Dataset'] == dataset]
        
        x = range(len(dataset_df))
        means = dataset_df['PPL_Mean'].values
        stds = dataset_df['PPL_Std'].values
        
        ax.errorbar(x, means, yerr=stds, label=dataset, 
                   marker='o', capsize=5, capthick=2)
    
    ax.set_xticks(range(len(dataset_df)))
    ax.set_xticklabels(dataset_df['Model'].values, rotation=45)
    ax.set_ylabel('Perplexity')
    ax.set_title('Model Performance Across Seeds (Mean ± Std)')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_dir / 'seed_variance.png', dpi=300)
    plt.close()


def plot_parameter_efficiency(tracker: MultiSeedMetricsTracker, save_dir: Path, 
                              model_params: Dict[str, int] = None):
    """
    Plot performance vs parameter count across seeds
    
    Args:
        tracker: Metrics tracker with multi-seed results
        save_dir: Directory to save plots
        model_params: Dict mapping model_name -> parameter_count
                     If None, will attempt to extract from saved models
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    df = tracker.get_aggregated_results()
    
    # if model_params is None:
    #     # Default parameter counts (update based on your actual models)
    #     model_params = {
    #         'baseline_512': 1,#26_214_400,  # 512 * 50257 (vocab size)
    #         'baseline_256': 1,#3_276_800,    # 64 * 50257
    #         'baseline_64': 1,#3_276_800,    # 64 * 50257
    #         'moe_dense_8x64': 1,#3_276_800 + 400_000,    # Smaller base + experts
    #         'moe_sparse_top8_8x64': 1,#26_214_400 + 500_000,  # Base + routing overhead
    #         'moe_expert_choice_8x64': 1,#3_276_800 + 400_000,    # Smaller base + experts
    #     }
    # REPLACE THIS BLOCK:
    if model_params is None:
        # Extract from tracker instead of hardcoding
        model_params = {}
        for dataset in tracker.metrics:
            for model in tracker.metrics[dataset]:
                seeds = tracker.metrics[dataset][model]
                first_seed = list(seeds.keys())[0]
                final = seeds[first_seed].get('final', {})
                
                # Get params (should be same across seeds)
                if 'total_params' in final:
                    model_params[model] = final['total_params']
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Group by dataset for different colors
    datasets = df['Dataset'].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(datasets)))
    
    for dataset, color in zip(datasets, colors):
        dataset_df = df[df['Dataset'] == dataset]
        
        params = []
        means = []
        stds = []
        labels = []
        
        for _, row in dataset_df.iterrows():
            model = row['Model']
            if model in model_params:
                params.append(model_params[model] / 1e6)  # Convert to millions
                means.append(row['PPL_Mean'])
                stds.append(row['PPL_Std'])
                labels.append(model)
        
        # Plot with error bars
        ax.errorbar(params, means, yerr=stds, 
                   fmt='o', label=dataset, color=color,
                   capsize=5, capthick=2, markersize=8, alpha=0.7)
        
        # Annotate points with model names
        for p, m, s, l in zip(params, means, stds, labels):
            ax.annotate(l, (p, m), textcoords="offset points",
                       xytext=(0, 10), ha='center', fontsize=8, alpha=0.7)
    
    ax.set_xlabel('Parameters (Millions)')
    ax.set_ylabel('Perplexity (Mean ± Std)')
    ax.set_title('Parameter Efficiency: Performance vs Model Size')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    
    plt.tight_layout()
    plt.savefig(save_dir / 'parameter_efficiency.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # Also create a table
    efficiency_data = []
    for dataset in datasets:
        dataset_df = df[df['Dataset'] == dataset]
        for _, row in dataset_df.iterrows():
            model = row['Model']
            if model in model_params:
                efficiency_data.append({
                    'Dataset': dataset,
                    'Model': model,
                    'Params (M)': model_params[model] / 1e6,
                    'PPL Mean': row['PPL_Mean'],
                    'PPL Std': row['PPL_Std'],
                    'PPL per M params': row['PPL_Mean'] / (model_params[model] / 1e6)
                })
    
    efficiency_df = pd.DataFrame(efficiency_data)
    efficiency_df.to_csv(save_dir / 'parameter_efficiency.csv', index=False)
    print(f"\nParameter efficiency table saved to {save_dir / 'parameter_efficiency.csv'}")


def plot_expert_usage(tracker: MultiSeedMetricsTracker, save_dir: Path):
    """
    Visualize expert usage patterns aggregated across seeds
    Shows mean usage with error bars representing variance across seeds
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Collect all MoE models with expert usage data
    moe_results = []
    
    for dataset in tracker.metrics:
        for model in tracker.metrics[dataset]:
            # Check if this is an MoE model by looking at any seed
            seeds = tracker.metrics[dataset][model]
            if not seeds:
                continue
            
            # Get first seed to check if expert usage exists
            first_seed = list(seeds.keys())[0]
            final = seeds[first_seed].get('final', {})
            
            if 'expert_usage_mean' in final or 'expert_usage' in final:
                moe_results.append((dataset, model))
    
    if not moe_results:
        print("No MoE models with expert usage data found. Skipping expert usage plot.")
        return
    
    # Create subplots
    n_plots = len(moe_results)
    n_cols = min(3, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    if n_plots == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for idx, (dataset, model) in enumerate(moe_results):
        ax = axes[idx]
        
        # Aggregate expert usage across seeds
        all_expert_usages = []
        
        for seed in tracker.metrics[dataset][model]:
            final = tracker.metrics[dataset][model][seed].get('final', {})
            usage = final.get('expert_usage_mean', final.get('expert_usage', None))
            
            if usage is not None:
                # Convert to numpy if needed
                if isinstance(usage, list):
                    usage = np.array(usage)
                elif hasattr(usage, 'cpu'):
                    usage = usage.cpu().numpy()
                all_expert_usages.append(usage)
        
        if not all_expert_usages:
            continue
        
        # Compute mean and std across seeds
        all_expert_usages = np.array(all_expert_usages)  # [num_seeds, num_experts]
        usage_mean = all_expert_usages.mean(axis=0)
        usage_std = all_expert_usages.std(axis=0)
        
        # Plot
        experts = range(len(usage_mean))
        bars = ax.bar(experts, usage_mean, yerr=usage_std, capsize=5, 
                     alpha=0.7, edgecolor='black', linewidth=1.5)
        
        # Color bars by usage level
        norm_usage = usage_mean / usage_mean.max() if usage_mean.max() > 0 else usage_mean
        colors = plt.cm.viridis(norm_usage)
        for bar, color in zip(bars, colors):
            bar.set_color(color)
        
        # Add horizontal line for uniform usage
        uniform_usage = 1.0 / len(usage_mean)
        ax.axhline(uniform_usage, color='red', linestyle='--', 
                  label=f'Uniform ({uniform_usage:.3f})', alpha=0.7)
        
        ax.set_xlabel('Expert ID', fontsize=11)
        ax.set_ylabel('Usage Frequency', fontsize=11)
        ax.set_title(f'{dataset}\n{model}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        
        # Add text annotation with stats
        usage_cv = usage_std.mean() / usage_mean.mean() if usage_mean.mean() > 0 else 0
        ax.text(0.02, 0.98, f'CV: {usage_cv:.3f}', 
               transform=ax.transAxes, fontsize=9,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Hide unused subplots
    for idx in range(n_plots, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(save_dir / 'expert_usage.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Expert usage plot saved to {save_dir / 'expert_usage.png'}")


def plot_comparison_results(tracker: MultiSeedMetricsTracker, save_dir: str,
                           model_params: Dict[str, int] = None):
    """
    Generate comprehensive comparison plots for paper
    
    Args:
        tracker: Multi-seed metrics tracker
        save_dir: Directory to save plots
        model_params: Optional dict of model -> parameter count
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    df = tracker.get_aggregated_results()
    
    if df.empty:
        print("No results to plot. Skipping visualization.")
        return
    
    # 1. Perplexity comparison across datasets with error bars
    fig, ax = plt.subplots(figsize=(14, 7))
    
    datasets = sorted(df['Dataset'].unique())
    models = sorted(df['Model'].unique())
    
    x = np.arange(len(datasets))
    width = 0.8 / len(models)  # Adjust width based on number of models
    
    for i, model in enumerate(models):
        means = []
        stds = []
        
        for dataset in datasets:
            dataset_model = df[(df['Dataset'] == dataset) & (df['Model'] == model)]
            if not dataset_model.empty:
                means.append(dataset_model['PPL_Mean'].values[0])
                stds.append(dataset_model['PPL_Std'].values[0])
            else:
                means.append(0)
                stds.append(0)
        
        # Plot bars with error bars
        bars = ax.bar(x + i * width, means, width, 
                     yerr=stds, capsize=4, label=model,
                     alpha=0.8, edgecolor='black', linewidth=1)
    
    ax.set_xlabel('Dataset', fontsize=12, fontweight='bold')
    ax.set_ylabel('Perplexity (Mean ± Std)', fontsize=12, fontweight='bold')
    ax.set_title('Model Comparison Across Datasets (Multi-Seed)', 
                fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels(datasets, rotation=45, ha='right')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_dir / 'perplexity_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Perplexity comparison saved to {save_dir / 'perplexity_comparison.png'}")
    
    # 2. Parameter efficiency plot
    plot_parameter_efficiency(tracker, save_dir, model_params)
    
    # 3. Expert usage visualization (for MoE models)
    plot_expert_usage(tracker, save_dir)
    
    # 4. Seed variance visualization
    plot_seed_variance(tracker, save_dir)

# # ============================================================================
# # MAIN EVALUATION SCRIPT
# # ============================================================================
# checkpoint_dir = "./checkpoints"
# os.makedirs(checkpoint_dir, exist_ok=True)
# def run_full_evaluation():
#     """Run complete evaluation suite"""
    
#     config = EvalConfig()
#     tracker = MetricsTracker('results')
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
#     print("=" * 80)
#     print("MoE Embeddings: Systematic Evaluation")
#     print("=" * 80)
    
#     # For each dataset
#     for dataset_name in tqdm(config.DATASETS):
#         print(f"\n{'=' * 80}")
#         print(f"Dataset: {dataset_name}")
#         print(f"{'=' * 80}\n")
        
#         # Load dataset (implement your data loading)
#         train_loader, val_loader, test_loader, vocab_size = load_dataset(dataset_name, config)
        
#         # For each model configuration
#         for model_name, model_config in config.MODEL_CONFIGS.items():
#             print(f"\nTraining {model_name}...")
            
#             # Build model (implement based on your architecture)
#             model = build_model(vocab_size, model_config,config ).to(device)
#             print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')
            
            
#             # Train and evaluate
#             results = train_and_evaluate(
#                 model, train_loader, val_loader, test_loader,
#                 config, tracker, dataset_name, model_name, device
#             )
            
#             # Save checkpoint
#             torch.save(
#                 model.state_dict(),
#                 f'{checkpoint_dir}/{dataset_name}_{model_name}.pt'
#             )
    
#     # Save all results
#     tracker.save()
    
#     # Generate comparison table
#     print("\n" + "=" * 80)
#     print("FINAL RESULTS")
#     print("=" * 80 + "\n")
#     display(tracker.get_comparison_table(format='dataframe'))
#     tracker.print_comparison()
    
#     # Generate plots
#     plot_comparison_results(tracker, 'results/figures')
    
#     print("\nEvaluation complete! Results saved to 'results/' directory")


# # ============================================================================
# # PAPER-READY RESULTS FORMATTER
# # ============================================================================

# def generate_latex_table(tracker: MetricsTracker, output_file: str = 'results_table.tex'):
#     """Generate LaTeX table for paper"""
    
#     datasets = set()
#     models = set()
#     for key in tracker.metrics.keys():
#         dataset, model = key.split('/')
#         datasets.add(dataset)
#         models.add(model)
    
#     lines = []
#     lines.append(r"\begin{table}[t]")
#     lines.append(r"\centering")
#     lines.append(r"\caption{Perplexity comparison across datasets}")
#     lines.append(r"\label{tab:results}")
#     lines.append(r"\begin{tabular}{l" + "c" * len(models) + "}")
#     lines.append(r"\toprule")
    
#     # Header
#     header = "Dataset & " + " & ".join(models) + r" \\"
#     lines.append(header)
#     lines.append(r"\midrule")
    
#     # Data rows
#     for dataset in sorted(datasets):
#         row = [dataset]
#         for model in sorted(models):
#             key = f"{dataset}/{model}"
#             if key in tracker.metrics and 'final' in tracker.metrics[key]:
#                 ppl = tracker.metrics[key]['final'].get('perplexity', float('inf'))
#                 row.append(f"{ppl:.2f}")
#             else:
#                 row.append("--")
#         lines.append(" & ".join(row) + r" \\")
    
#     lines.append(r"\bottomrule")
#     lines.append(r"\end{tabular}")
#     lines.append(r"\end{table}")
    
#     latex = "\n".join(lines)
    
#     with open(output_file, 'w') as f:
#         f.write(latex)
    
#     return latex


# ============================================================================
# HELPER FUNCTIONS (TO BE IMPLEMENTED)
# ============================================================================



def build_model(vocab_size: int, model_config: Dict, config):
    # print(model_config)

    model = BigramLanguageModel(vocab_size, config.SEQ_LEN, **model_config)
    """Build model from configuration - integrate with your TrueSoftRoutingEmbeddings"""
    # Placeholder - implement based on your model architecture
    return model# NotImplementedError("Implement model building with your MoE embeddings")


# if __name__ == "__main__":
#     # Run evaluation
#     run_full_evaluation()
    
#     # Generate paper-ready outputs
#     tracker = MetricsTracker('results')
#     tracker.load()  # Implement load method
    
#     print("\nGenerating LaTeX table...")
#     latex_table = generate_latex_table(tracker)
#     print(latex_table)


if __name__ == "__main__":
    tracker = run_multi_seed_evaluation()
    plot_seed_variance(tracker, 'results/figures')

Multi-Seed Evaluation (1 seeds)

Dataset: wikitext-103

--------------------------------------------------------------------------------
Model: moe_remixed_lin
--------------------------------------------------------------------------------

Loading wikitext-103...
Building vocabulary...

  Seed 1/1 (seed=42)


Seed 42:   0%|          | 0/5000 [00:00<?, ?it/s]

[Seed 42] Step 0: val_ppl=55983.14
[Seed 42] Step 1000: val_ppl=290.39
[Seed 42] Step 2000: val_ppl=158.94
[Seed 42] Step 3000: val_ppl=103.90
[Seed 42] Step 4000: val_ppl=68.71
    Test PPL: 55.63

AGGREGATED RESULTS (Mean ± Std)

┌─────────────────┬──────────────────────┐
│ Dataset         │ moe_remixed_lin      │
├─────────────────┼──────────────────────┤
│ wikitext-103    │ 55.63 ± 0.00         │
└─────────────────┴──────────────────────┘

STATISTICAL SIGNIFICANCE TESTS

Empty DataFrame
Columns: []
Index: []

LaTeX table saved to: results/results_table.tex
Perplexity comparison saved to results/figures/perplexity_comparison.png

Parameter efficiency table saved to results/figures/parameter_efficiency.csv
No MoE models with expert usage data found. Skipping expert usage plot.

Evaluation complete! Results saved to 'results/' directory


In [11]:
ABLATIONS = {
    "full": dict(use_basis_gate=True, use_output_gate=True, use_context=True),
    "no_basis_gate": dict(use_basis_gate=False, use_output_gate=True, use_context=True),
    "no_output_gate": dict(use_basis_gate=True, use_output_gate=False, use_context=True),
    "no_context": dict(use_basis_gate=True, use_output_gate=True, use_context=False),
    "static_linear": dict(use_basis_gate=False, use_output_gate=False, use_context=False),
}


In [12]:

base_config = {
        
            'use_moe': False,
            'total_embed_dim': 64,
            'use_remixed_linear':True,
        }
def build_model_with_ablation(base_config, ablation_cfg, config):
    cfg = base_config
    cfg['remixed_linear_kwargs'] = ablation_cfg

    return build_model(vocab_size, cfg, config)



seed = 42


In [13]:
config = EvalConfig()
train_loader, val_loader, test_loader, vocab_size = load_dataset(
    "wikitext-103", config,42)
def run_ablation_experiment(
    ABLATIONS,
    base_config,
    config, 
    build_model_fn,
    train_loader,
    val_loader,
    device,
    seed,
):
    all_results = {}

    for model_name, ablation_cfg in ABLATIONS.items():
        print(f"\n=== Ablation: {model_name} ===")

        # ---- Reproducibility ----
        set_seed(seed)

        # ---- Build model ----
        model = build_model_fn(base_config, ablation_cfg, config)
        model = model.to(device)

        # ---- Parameter counts ----
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )

        print(f"Total params: {total_params:,}")
        print(f"Trainable params: {trainable_params:,}")

        # ---- Optimizer ----
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.LEARNING_RATE *config.MOE_SCALE ,
            weight_decay=0.01,
        )

        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=config.LEARNING_RATE*config.MOE_SCALE,
            total_steps=config.MAX_ITERS,
            pct_start=config.WARMUP_PCT,
        )

        model.train()
        best_val_loss = float("inf")
        train_iter = iter(train_loader)

        train_losses = []
        grad_norms = []

        for step in tqdm(
            range(config.MAX_ITERS),
            desc=f"{model_name} | Seed {seed}",
        ):
            try:
                x, y = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                x, y = next(train_iter)

            x, y = x.to(device), y.to(device)

            # ---- Forward ----
            loss, total_loss, info = model(x, y)

            # ---- Backward ----
            optimizer.zero_grad()
            total_loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            optimizer.step()
            scheduler.step()

            train_losses.append(loss.item())
            grad_norms.append(grad_norm.item())

            # ---- Evaluation ----
            if (
                step % config.EVAL_INTERVAL == 0
                or step == config.MAX_ITERS - 1
            ):
                val_results = evaluate_model(
                    model,
                    val_loader,
                    device,
                    config.EVAL_ITERS,
                )

                if val_results["loss"] < best_val_loss:
                    best_val_loss = val_results["loss"]

                tqdm.write(
                    f"[{model_name}] "
                    f"step={step} "
                    f"train_loss={loss.item():.4f} "
                    f"val_loss={val_results['loss']:.4f} "
                    f"ppl={val_results['perplexity']:.2f}"
                )

        # ---- Store results ----
        all_results[model_name] = {
            "best_val_loss": best_val_loss,
            "final_train_loss": np.mean(train_losses[-100:]),
            "perplexity": np.exp(best_val_loss),
            "grad_norm_mean": float(np.mean(grad_norms)),
            "grad_norm_std": float(np.std(grad_norms)),
            "total_params": total_params,
            "trainable_params": trainable_params,
        }

    return all_results



Loading wikitext-103...
Building vocabulary...


In [14]:
run_ablation_experiment(
    ABLATIONS,
    base_config,
    config, 
    build_model_with_ablation,
    train_loader,
    val_loader,
    device,
    seed,
)


=== Ablation: full ===
Total params: 10,286,419
Trainable params: 10,286,419


full | Seed 42:   0%|          | 0/5000 [00:00<?, ?it/s]

[full] step=0 train_loss=10.9651 val_loss=10.9328 ppl=55983.14
[full] step=1000 train_loss=5.6416 val_loss=5.6712 ppl=290.39
[full] step=2000 train_loss=5.5051 val_loss=5.0685 ppl=158.94
[full] step=3000 train_loss=4.7234 val_loss=4.6435 ppl=103.90
[full] step=4000 train_loss=4.3288 val_loss=4.2299 ppl=68.71
[full] step=4999 train_loss=3.9680 val_loss=4.0807 ppl=59.19

=== Ablation: no_basis_gate ===
Total params: 10,286,419
Trainable params: 10,286,419


no_basis_gate | Seed 42:   0%|          | 0/5000 [00:00<?, ?it/s]

[no_basis_gate] step=0 train_loss=10.9635 val_loss=10.9295 ppl=55797.63
[no_basis_gate] step=1000 train_loss=5.6596 val_loss=5.6605 ppl=287.30
[no_basis_gate] step=2000 train_loss=5.5174 val_loss=5.0818 ppl=161.06
[no_basis_gate] step=3000 train_loss=4.7863 val_loss=4.7082 ppl=110.85
[no_basis_gate] step=4000 train_loss=4.4832 val_loss=4.3862 ppl=80.34
[no_basis_gate] step=4999 train_loss=4.1545 val_loss=4.2484 ppl=69.99

=== Ablation: no_output_gate ===
Total params: 10,286,419
Trainable params: 10,286,419


...

no_context | Seed 42:   0%|          | 0/5000 [00:00<?, ?it/s]

[no_context] step=0 train_loss=10.9624 val_loss=10.9285 ppl=55740.78
[no_context] step=1000 train_loss=6.0934 val_loss=6.0841 ppl=438.83
[no_context] step=2000 train_loss=6.0357 val_loss=5.5860 ppl=266.66
[no_context] step=3000 train_loss=5.3764 val_loss=5.2655 ppl=193.54
[no_context] step=4000 train_loss=5.1893 val_loss=5.0808 ppl=160.90
[no_context] step=4999 train_loss=4.8736 val_loss=5.0198 ppl=151.39

=== Ablation: static_linear ===
Total params: 10,286,419
Trainable params: 10,286,419


static_linear | Seed 42:   0%|          | 0/5000 [00:00<?, ?it/s]

[static_linear] step=0 train_loss=10.9624 val_loss=10.9285 ppl=55740.78
[static_linear] step=1000 train_loss=6.0934 val_loss=6.0841 ppl=438.83
[static_linear] step=2000 train_loss=6.0357 val_loss=5.5860 ppl=266.66
[static_linear] step=3000 train_loss=5.3764 val_loss=5.2655 ppl=193.54
[static_linear] step=4000 train_loss=5.1893 val_loss=5.0808 ppl=160.90
[static_linear] step=4999 train_loss=4.8736 val_loss=5.0198 ppl=151.39


{'full': {'best_val_loss': np.float64(4.080711498260498),
  'final_train_loss': np.float64(4.165826201438904),
  'perplexity': np.float64(59.18756672294686),
  'grad_norm_mean': 0.9441099766969681,
  'grad_norm_std': 0.41223013147465365,
  'total_params': 10286419,
  'trainable_params': 10286419},
 'no_basis_gate': {'best_val_loss': np.float64(4.2483501923084255),
  'final_train_loss': np.float64(4.338200554847718),
  'perplexity': np.float64(69.989847254479),
  'grad_norm_mean': 0.756249682444334,
  'grad_norm_std': 0.19883086106230996,
  'total_params': 10286419,
  'trainable_params': 10286419},
 'no_output_gate': {'best_val_loss': np.float64(4.151694486141205),
  'final_train_loss': np.float64(4.237778151035309),
  'perplexity': np.float64(63.54157945246187),
  'grad_norm_mean': 0.9045231689631938,
  'grad_norm_std': 0.39637037739845643,
  'total_params': 10286419,
  'trainable_params': 10286419},
 'no_context': {'best_val_loss': np.float64(5.019827363491058),
  'final_train_loss': 

In [15]:
# config = EvalConfig()
# model_config = config.MODEL_CONFIGS['moe_dense_8x64_no_replacement_soft']
# model_config
# model = BigramLanguageModel(vocab_size, 64, **model_config)
# for name, param in model.named_parameters():
#     if any(x in name for x in ['embedding_model']): #'dim_selectors', 'expert_mlps', 'expert_mlp' 'embeddings', 'expert_router']):
#         print(name)

In [16]:
#         'moe_dense_8x64': {
#             'use_moe': True,
#             'total_embed_dim': 512,
#             'use_shared_base': False,
#             'shared_base_dim': 128,
#             'expert_residual':False,  # Add residual connections in experts
#             'use_vocab_prior':False,
#             'use_expert_bias':False,
            
#             'num_experts': 8,
#             'use_sparse_top_k': False,
#             'routing_mode': 'token_choice'
#         },
# [wikitext-103/moe_dense_8x64] Iter 0: train_loss=10.9975, val_loss=10.9683, val_ppl=58006.98
# [wikitext-103/moe_dense_8x64] Iter 1000: train_loss=6.5295, val_loss=6.3036, val_ppl=546.52
# [wikitext-103/moe_dense_8x64] Iter 2000: train_loss=5.8992, val_loss=5.7266, val_ppl=306.92
# [wikitext-103/moe_dense_8x64] Iter 3000: train_loss=5.6263, val_loss=5.4406, val_ppl=230.59
# [wikitext-103/moe_dense_8x64] Iter 4000: train_loss=5.4450, val_loss=5.2768, val_ppl=195.74
# [wikitext-103/moe_dense_8x64] Iter 4999: train_loss=5.5823, val_loss=5.2782, val_ppl=196.01

# [wikitext-103/moe_dense_8x64] Final Test Results:
#   Perplexity: 177.79
#   Loss: 5.1806

#         'moe_dense_8x64': {
#             'use_moe': True,
#             'total_embed_dim': 512,
#             'use_shared_base': True,
#             'shared_base_dim': 128,
#             'expert_residual':False,  # Add residual connections in experts
#             'use_vocab_prior':False,
#             'use_expert_bias':False,
            
#             'num_experts': 8,
#             'use_sparse_top_k': False,
#             'routing_mode': 'token_choice'
#         },

In [17]:
# # [wikitext-103/baseline_512] Iter 4000: train_loss=4.8586, val_ppl=105.02
# # [wikitext-103/baseline_512] Iter 4500: train_loss=4.6664, val_ppl=97.55
# # [wikitext-103/baseline_512] Iter 4999: train_loss=4.4383, val_ppl=98.77

# Training baseline_512...
# 70.462545 M parameters

# 100%
#  5000/5000 [08:59<00:00,  1.24s/it]

# [wikitext-103/baseline_512] Iter 0: train_loss=10.9960, val_loss=10.8657, val_ppl=52349.83
# [wikitext-103/baseline_512] Iter 1000: train_loss=5.7706, val_loss=5.8369, val_ppl=342.71
# [wikitext-103/baseline_512] Iter 2000: train_loss=5.2348, val_loss=5.2423, val_ppl=189.11
# [wikitext-103/baseline_512] Iter 3000: train_loss=4.7374, val_loss=4.8730, val_ppl=130.71
# [wikitext-103/baseline_512] Iter 4000: train_loss=4.7438, val_loss=4.6401, val_ppl=103.56
# [wikitext-103/baseline_512] Iter 4999: train_loss=4.7315, val_loss=4.5685, val_ppl=96.40

# [wikitext-103/baseline_512] Final Test Results:
#   Perplexity: 91.74
#   Loss: 4.5190

# Training baseline_64...
# 6.787409 M parameters

# 100%
#  5000/5000 [02:09<00:00, 40.30it/s]

# [wikitext-103/baseline_64] Iter 0: train_loss=10.9534, val_loss=10.9461, val_ppl=56733.70
# [wikitext-103/baseline_64] Iter 1000: train_loss=6.6680, val_loss=6.5755, val_ppl=717.27
# [wikitext-103/baseline_64] Iter 2000: train_loss=6.0566, val_loss=5.9495, val_ppl=383.57
# [wikitext-103/baseline_64] Iter 3000: train_loss=5.7164, val_loss=5.6817, val_ppl=293.46
# [wikitext-103/baseline_64] Iter 4000: train_loss=5.5989, val_loss=5.5457, val_ppl=256.14
# [wikitext-103/baseline_64] Iter 4999: train_loss=5.6078, val_loss=5.4984, val_ppl=244.30

# [wikitext-103/baseline_64] Final Test Results:
#   Perplexity: 243.32
#   Loss: 5.4944

# Training baseline_256...
# 30.537809 M parameters

# 100%
#  5000/5000 [04:45<00:00,  2.96it/s]

# [wikitext-103/baseline_256] Iter 0: train_loss=10.9807, val_loss=10.9157, val_ppl=55034.17
# [wikitext-103/baseline_256] Iter 1000: train_loss=6.0835, val_loss=5.9861, val_ppl=397.86
# [wikitext-103/baseline_256] Iter 2000: train_loss=5.2044, val_loss=5.4119, val_ppl=224.05
# [wikitext-103/baseline_256] Iter 3000: train_loss=5.2157, val_loss=5.0827, val_ppl=161.21
# [wikitext-103/baseline_256] Iter 4000: train_loss=5.0174, val_loss=4.8472, val_ppl=127.38
# [wikitext-103/baseline_256] Iter 4999: train_loss=4.9881, val_loss=4.7878, val_ppl=120.04

# [wikitext-103/baseline_256] Final Test Results:
#   Perplexity: 116.41
#   Loss: 4.7571